# EXP 04B — Train-Only Signal Distillation dengan Directional Tail/Blowout Awareness, di Atas History-Full Freeze Backbone yang Submission-Safe

Notebook ini adalah **branch distillation** yang dibangun di atas backbone terbaik yang sudah relatif stabil, yaitu **history-full + freeze-safe inference**.  
Fokusnya bukan membuka branch probabilistik atau recursive lagi, tetapi mencoba memindahkan sebagian sinyal **train-only official features** ke student yang tetap **submission-safe**.

Tujuan praktis EXP04B ada tiga:

1. mengukur **headroom** dari teacher yang boleh memakai fitur train-only resmi,
2. memindahkan sinyal teacher itu ke student lewat **distillation yang fair**,
3. menambahkan **directional blowout awareness** agar student lebih peka terhadap arah kemenangan besar, bukan hanya “match ini ekstrem atau tidak”.

Notebook ini dijaga tetap:

- **standalone**
- **freeze-safe**
- **tanpa external data**
- **tanpa membaca output eksperimen lama sebagai dependency**
- dan **tanpa grid tuning yang meledak**


## 01. Setup, Seed, Path, dan Guardrail Runtime

Di tahap ini kita menyiapkan environment eksperimen, path input/output, dan beberapa guardrail penting agar EXP04B tetap terkendali.

Guardrail utama notebook ini:

- teacher hanya untuk **analysis/training**, **bukan** untuk submission test,
- jumlah **OOF temporal blocks maksimal 4**,
- decoder tuning dibuat **raw-output-first**,
- grid directional tail dibuat **staged**, bukan brute-force besar,
- inference final tetap **freeze-safe only**.


In [ ]:
import os
import json
import math
import copy
import random
import warnings
from pathlib import Path
from collections import Counter, deque
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, mean_absolute_error, log_loss
from catboost import CatBoostRegressor, CatBoostClassifier, Pool

warnings.filterwarnings("ignore")

SEED = 42

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

TRAIN_PATH = Path("../data/train.csv")
TEST_PATH = Path("../data/test.csv")
SAMPLE_SUB_PATH = Path("../data/sample submission.csv")
META_PATH = Path("../data/metadata.txt")

OUT_ROOT = Path("../outputs/exp04b_train_only_distill_directional_tail")
FIG_DIR = OUT_ROOT / "figures"
PRED_DIR = OUT_ROOT / "predictions"
SUB_DIR = OUT_ROOT / "submissions"
SUM_DIR = OUT_ROOT / "summaries"

for folder in [OUT_ROOT, FIG_DIR, PRED_DIR, SUB_DIR, SUM_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[OK] Output root siap: {OUT_ROOT}")


In [ ]:
def seed_everything(seed: int = 42) -> None:
    """Set seed dasar untuk reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
print(f"[OK] Seed aktif: {SEED}")


### Guardrail eksperimen

- Notebook ini **standalone** dan membangun ulang seluruh pipeline dari input mentah.
- Teacher **boleh** memakai fitur train-only, tetapi **tidak boleh** dipakai langsung untuk submission test.
- Student final **harus** hanya memakai fitur legal/test-feasible.
- Teacher OOF default memakai **3 temporal blocks** dan maksimal **4**.
- Decoder directional-tail memakai **staged tuning**, bukan grid raksasa.


## 02. Validasi File Input

Sebelum pipeline utama dijalankan, kita cek dulu apakah semua file input kompetisi tersedia dan ukurannya masuk akal.


In [ ]:
def validate_input_files(file_paths: dict) -> None:
    missing = []
    print("=== INPUT FILE CHECK ===")
    for name, path in file_paths.items():
        exists = Path(path).exists()
        size_mb = Path(path).stat().st_size / (1024 * 1024) if exists else np.nan
        print(
            f"- {name:<15}: {path} | exists={exists} | size_mb={size_mb:.4f}"
            if exists else
            f"- {name:<15}: {path} | exists={exists}"
        )
        if not exists:
            missing.append(str(path))
    if missing:
        raise FileNotFoundError("File input berikut tidak ditemukan: " + ", ".join(missing))
    print("[OK] Semua file input ditemukan.")

validate_input_files(
    {
        "train": TRAIN_PATH,
        "test": TEST_PATH,
        "sample_submission": SAMPLE_SUB_PATH,
        "metadata": META_PATH,
    }
)


## 03. Load Data dan Helper Dasar

Di bagian ini kita:

- load seluruh file input,
- parse `date`,
- dan membawa kembali helper inti yang dibutuhkan dari fondasi eksperimen sebelumnya:
  - canonical match builder
  - evaluator AW-MAE
  - reverse mapping ke submission
  - temporal holdout leakage-safe


In [ ]:
train = pd.read_csv(TRAIN_PATH, parse_dates=["date"])
test = pd.read_csv(TEST_PATH, parse_dates=["date"])
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    metadata_text = f.read()

print("[INFO] train shape             :", train.shape)
print("[INFO] test shape              :", test.shape)
print("[INFO] sample_submission shape :", sample_submission.shape)
print("[INFO] jumlah kolom train      :", len(train.columns))
print("[INFO] jumlah kolom test       :", len(test.columns))
print("[INFO] metadata chars          :", len(metadata_text))

display(train.head(3))
display(test.head(3))
display(sample_submission.head(3))


In [ ]:
def _safe_string_series(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("__MISSING__").astype(str)

def _safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    out = a / b.replace(0, np.nan)
    out = out.replace([np.inf, -np.inf], np.nan)
    return out

def _mean_or_nan(values):
    values = list(values)
    return float(np.mean(values)) if len(values) > 0 else np.nan

def _rate_from_bool_list(values):
    values = list(values)
    return float(np.mean(values)) if len(values) > 0 else np.nan

def _outcome(a: int, b: int) -> int:
    if a > b:
        return 0
    if a == b:
        return 1
    return 2

EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50

def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).strip().lower()
    if ("fifa world cup" in t) or (t == "world cup"):
        return 2.00
    if ("afc championship" in t) or ("afc asian cup" in t) or ("asian cup" in t):
        return 1.80
    if "friendly" in t:
        return 0.96
    return 1.20

def official_match_loss(y_team_true: int, y_opp_true: int, y_team_pred: int, y_opp_pred: int) -> float:
    y_team_true = int(y_team_true)
    y_opp_true = int(y_opp_true)
    y_team_pred = int(y_team_pred)
    y_opp_pred = int(y_opp_pred)

    mae = (abs(y_team_true - y_team_pred) + abs(y_opp_true - y_opp_pred)) / 2.0

    exact = int((y_team_true == y_team_pred) and (y_opp_true == y_opp_pred))
    outcome_true = _outcome(y_team_true, y_opp_true)
    outcome_pred = _outcome(y_team_pred, y_opp_pred)
    outcome_ok = int(outcome_true == outcome_pred)

    gd_true = y_team_true - y_opp_true
    gd_pred = y_team_pred - y_opp_pred
    gd_ok = int(gd_true == gd_pred)

    penalty = (
        EXACT_PENALTY * (1 - exact)
        + OUTCOME_PENALTY * (1 - outcome_ok)
        + GD_PENALTY * (1 - gd_ok)
    )

    raw_loss = mae + penalty
    multiplier = 1.0 if outcome_ok else WRONG_OUTCOME_MULTIPLIER
    return float((raw_loss * multiplier) ** NONLINEAR_POWER)

def awmae_score(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments) -> float:
    y_team_true = pd.Series(y_team_true).reset_index(drop=True)
    y_opp_true = pd.Series(y_opp_true).reset_index(drop=True)
    y_team_pred = pd.Series(y_team_pred).reset_index(drop=True)
    y_opp_pred = pd.Series(y_opp_pred).reset_index(drop=True)
    tournaments = pd.Series(tournaments).reset_index(drop=True)

    weights = tournaments.map(get_tournament_weight).astype(float)
    losses = [
        official_match_loss(a, b, pa, pb)
        for a, b, pa, pb in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred)
    ]
    losses = np.asarray(losses, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(np.sum(losses * weights) / np.sum(weights))

def make_time_based_holdout(train_match: pd.DataFrame, valid_fraction: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = train_match.sort_values(["date", "match_id"]).reset_index(drop=True).copy()
    n_valid = max(1, int(len(df) * valid_fraction))
    split_idx = len(df) - n_valid
    train_fold = df.iloc[:split_idx].reset_index(drop=True)
    valid_fold = df.iloc[split_idx:].reset_index(drop=True)
    assert set(train_fold["match_id"]).isdisjoint(set(valid_fold["match_id"]))
    return train_fold, valid_fold


In [ ]:
def build_match_level(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    """Build match-level canonical dataframe dari representasi dua-row-per-match."""
    required_cols = ["match_id", "team", "opponent", "date"]
    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        raise KeyError(f"Kolom wajib untuk canonicalization tidak ada: {missing_required}")

    group_sizes = df.groupby("match_id").size()
    bad_match_ids = group_sizes[group_sizes != 2]
    if len(bad_match_ids) > 0:
        raise ValueError(f"Ada match_id yang tidak punya tepat 2 row. Contoh: {bad_match_ids.head().to_dict()}")

    meta_cols = ["match_id", "date", "gender", "tournament", "venue_country", "neutral"]
    row_id_cols = ["Id"]
    handled_team_cols = [
        "team", "opponent", "is_home",
        "confederation_team", "confederation_opp",
        "population_team", "population_opp",
        "gdp_per_capita_team", "gdp_per_capita_opp",
        "distance_travel_team", "distance_travel_opp",
        "temperature_venue", "altitude_venue",
    ]
    target_cols = ["team_goals", "opp_goals"]

    ignore_cols = set(meta_cols + row_id_cols + handled_team_cols + target_cols)
    extra_side_cols = [c for c in df.columns if c not in ignore_cols]

    rows = []
    for match_id, grp in tqdm(df.groupby("match_id", sort=False), total=df["match_id"].nunique(), desc="build_match_level"):
        pair = grp.copy()
        sort_cols = []
        if "team" in pair.columns:
            sort_cols.append("team")
        if "Id" in pair.columns:
            sort_cols.append("Id")
        if len(sort_cols) > 0:
            pair = pair.sort_values(sort_cols).reset_index(drop=True)
        row_a = pair.iloc[0]
        row_b = pair.iloc[1]

        out = {"match_id": match_id}
        for c in meta_cols:
            if c in pair.columns:
                out[c] = row_a[c]

        out["id_a"] = row_a["Id"] if "Id" in pair.columns else np.nan
        out["id_b"] = row_b["Id"] if "Id" in pair.columns else np.nan

        out["team_a"] = row_a["team"]
        out["team_b"] = row_b["team"]
        out["team_a_is_home"] = row_a["is_home"] if "is_home" in pair.columns else np.nan
        out["team_b_is_home"] = row_b["is_home"] if "is_home" in pair.columns else np.nan

        out["team_a_confederation"] = row_a["confederation_team"] if "confederation_team" in pair.columns else np.nan
        out["team_b_confederation"] = row_b["confederation_team"] if "confederation_team" in pair.columns else np.nan

        out["team_a_population"] = row_a["population_team"] if "population_team" in pair.columns else np.nan
        out["team_b_population"] = row_b["population_team"] if "population_team" in pair.columns else np.nan

        out["team_a_gdp_per_capita"] = row_a["gdp_per_capita_team"] if "gdp_per_capita_team" in pair.columns else np.nan
        out["team_b_gdp_per_capita"] = row_b["gdp_per_capita_team"] if "gdp_per_capita_team" in pair.columns else np.nan

        out["team_a_distance_travel"] = row_a["distance_travel_team"] if "distance_travel_team" in pair.columns else np.nan
        out["team_b_distance_travel"] = row_b["distance_travel_team"] if "distance_travel_team" in pair.columns else np.nan

        out["temperature_venue"] = row_a["temperature_venue"] if "temperature_venue" in pair.columns else np.nan
        out["altitude_venue"] = row_a["altitude_venue"] if "altitude_venue" in pair.columns else np.nan

        if is_train:
            out["team_a_goals"] = row_a["team_goals"]
            out["team_b_goals"] = row_b["team_goals"]

        for c in extra_side_cols:
            out[f"{c}_a"] = row_a[c]
            out[f"{c}_b"] = row_b[c]

        rows.append(out)

    match_df = pd.DataFrame(rows).sort_values(["date", "match_id"]).reset_index(drop=True)
    return match_df

def match_predictions_to_submission(
    test_row_df: pd.DataFrame,
    pred_match_df: pd.DataFrame,
    canonical_team_a_col: str = "team_a",
    canonical_team_b_col: str = "team_b",
    pred_a_col: str = "pred_team_a_goals",
    pred_b_col: str = "pred_team_b_goals",
) -> pd.DataFrame:
    merge_cols = ["match_id", canonical_team_a_col, canonical_team_b_col, pred_a_col, pred_b_col]
    tmp = test_row_df[["Id", "match_id", "team"]].merge(
        pred_match_df[merge_cols],
        on="match_id",
        how="left",
    )
    team_goals = np.where(
        tmp["team"] == tmp[canonical_team_a_col],
        tmp[pred_a_col],
        tmp[pred_b_col],
    )
    opp_goals = np.where(
        tmp["team"] == tmp[canonical_team_a_col],
        tmp[pred_b_col],
        tmp[pred_a_col],
    )

    submission = pd.DataFrame(
        {
            "Id": tmp["Id"],
            "team_goals": pd.Series(team_goals).astype(float).round().astype(int),
            "opp_goals": pd.Series(opp_goals).astype(float).round().astype(int),
        }
    )
    return submission


## 04. Cleaning Awal dan Canonical Match Data

Cleaning di sini tetap konservatif:

- `altitude_venue == -9999` diubah menjadi `NaN`,
- kolom string diamankan,
- lalu data diubah dari dua-row-per-match menjadi satu representasi canonical match-level.

Kita sengaja menjaga canonicalization tetap konsisten agar perbandingan antar eksperimen tetap fair.


In [ ]:
for df_name, df in [("train", train), ("test", test)]:
    if "altitude_venue" in df.columns:
        sentinel_count = int((df["altitude_venue"] == -9999).sum())
        df.loc[df["altitude_venue"] == -9999, "altitude_venue"] = np.nan
        print(f"[INFO] {df_name}: sentinel altitude -9999 ditemukan = {sentinel_count}")

string_cols = [
    "team", "opponent", "gender", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]
for col in string_cols:
    if col in train.columns:
        train[col] = _safe_string_series(train[col])
    if col in test.columns:
        test[col] = _safe_string_series(test[col])

train_match_base = build_match_level(train, is_train=True)
test_match_base = build_match_level(test, is_train=False)

print("[INFO] train_match_base shape:", train_match_base.shape)
print("[INFO] test_match_base shape :", test_match_base.shape)
display(train_match_base.head(3))
display(test_match_base.head(3))


## 05. Static + History-Full Legal Feature Pipeline

Di bagian ini kita membangun dua blok fitur student:

1. **static shared features** yang memang legal di train dan test,
2. **history-full legal features** yang dibangun leakage-safe hanya dari histori match sebelumnya.

Student final hanya boleh memakai kombinasi dua blok fitur ini.


In [ ]:
def engineer_static_match_features(df_match: pd.DataFrame) -> pd.DataFrame:
    df = df_match.copy()

    for col in ["team_a", "team_b", "gender", "tournament", "venue_country", "team_a_confederation", "team_b_confederation"]:
        if col in df.columns:
            df[col] = _safe_string_series(df[col])

    df["pair_key"] = df["team_a"] + "__VS__" + df["team_b"]
    df["confed_pair_key"] = df["team_a_confederation"] + "__VS__" + df["team_b_confederation"]

    df["match_year"] = df["date"].dt.year
    df["match_month"] = df["date"].dt.month
    df["match_quarter"] = df["date"].dt.quarter
    df["match_dayofweek"] = df["date"].dt.dayofweek
    df["match_dayofyear"] = df["date"].dt.dayofyear
    df["match_is_weekend"] = df["match_dayofweek"].isin([5, 6]).astype(int)
    df["match_decade"] = (df["match_year"] // 10) * 10

    df["home_side"] = np.select(
        [
            df["team_a_is_home"].fillna(0).astype(float) > df["team_b_is_home"].fillna(0).astype(float),
            df["team_a_is_home"].fillna(0).astype(float) < df["team_b_is_home"].fillna(0).astype(float),
        ],
        [1, -1],
        default=0,
    ).astype(int)

    df["same_confederation"] = (df["team_a_confederation"] == df["team_b_confederation"]).astype(int)

    t_lower = df["tournament"].str.lower()
    df["is_friendly"] = t_lower.str.contains("friendly").astype(int)
    df["is_world_cup"] = t_lower.str.contains("world cup").astype(int)
    df["is_qualification"] = t_lower.str.contains("qual").astype(int)
    df["is_nations_league"] = t_lower.str.contains("nations league").astype(int)
    df["tournament_weight_proxy"] = df["tournament"].map(get_tournament_weight).astype(float)

    numeric_pairs = [
        ("team_a_population", "team_b_population", "population"),
        ("team_a_gdp_per_capita", "team_b_gdp_per_capita", "gdp_pc"),
        ("team_a_distance_travel", "team_b_distance_travel", "travel"),
    ]

    for col_a, col_b, stem in numeric_pairs:
        df[col_a] = pd.to_numeric(df[col_a], errors="coerce")
        df[col_b] = pd.to_numeric(df[col_b], errors="coerce")

        df[f"{stem}_diff"] = df[col_a] - df[col_b]
        df[f"{stem}_abs_diff"] = (df[col_a] - df[col_b]).abs()

        df[f"log1p_{stem}_a"] = np.log1p(df[col_a].clip(lower=0))
        df[f"log1p_{stem}_b"] = np.log1p(df[col_b].clip(lower=0))
        df[f"log1p_{stem}_diff"] = df[f"log1p_{stem}_a"] - df[f"log1p_{stem}_b"]
        df[f"{stem}_ratio_ab"] = _safe_divide(df[col_a], df[col_b])

    df["temperature_venue"] = pd.to_numeric(df["temperature_venue"], errors="coerce")
    df["altitude_venue"] = pd.to_numeric(df["altitude_venue"], errors="coerce")

    return df


In [ ]:
def init_team_state() -> dict:
    return {
        "matches_played": 0,
        "last_match_date": pd.NaT,
        "elo_overall": 1500.0,
        "elo_goal_diff": 0.0,
        "ewm_points": 1.0,
        "ewm_gf": 1.2,
        "ewm_ga": 1.2,
        "ewm_gd": 0.0,
        "points_last5": deque(maxlen=5),
        "points_last10": deque(maxlen=10),
        "gf_last5": deque(maxlen=5),
        "ga_last5": deque(maxlen=5),
        "gd_last5": deque(maxlen=5),
        "result_last10": deque(maxlen=10),
        "clean_sheet_last5": deque(maxlen=5),
        "failed_to_score_last5": deque(maxlen=5),
    }

def init_h2h_state() -> dict:
    return {
        "matches_played": 0,
        "points_a_last3": deque(maxlen=3),
        "gd_a_last3": deque(maxlen=3),
        "total_last3": deque(maxlen=3),
    }

def expected_elo_result(rating_a: float, rating_b: float, home_bonus: float = 0.0) -> float:
    return 1.0 / (1.0 + 10.0 ** (-((rating_a + home_bonus - rating_b) / 400.0)))

def snapshot_team_features_full(state: dict, match_date: pd.Timestamp) -> dict:
    last_date = state["last_match_date"]
    if pd.isna(last_date):
        days_since_last = np.nan
    else:
        days_since_last = float((match_date - last_date).days)

    return {
        "hist_matches_played": float(state["matches_played"]),
        "hist_elo_overall": float(state["elo_overall"]),
        "hist_elo_gd": float(state["elo_goal_diff"]),
        "hist_ewm_points": float(state["ewm_points"]),
        "hist_ewm_gf": float(state["ewm_gf"]),
        "hist_ewm_ga": float(state["ewm_ga"]),
        "hist_ewm_gd": float(state["ewm_gd"]),
        "hist_points_avg_last5": _mean_or_nan(state["points_last5"]),
        "hist_points_avg_last10": _mean_or_nan(state["points_last10"]),
        "hist_gf_avg_last5": _mean_or_nan(state["gf_last5"]),
        "hist_ga_avg_last5": _mean_or_nan(state["ga_last5"]),
        "hist_gd_avg_last5": _mean_or_nan(state["gd_last5"]),
        "hist_win_rate_last10": _rate_from_bool_list([int(x == 3) for x in state["points_last10"]]),
        "hist_draw_rate_last10": _rate_from_bool_list([int(x == 1) for x in state["points_last10"]]),
        "hist_loss_rate_last10": _rate_from_bool_list([int(x == 0) for x in state["points_last10"]]),
        "hist_clean_sheet_rate_last5": _rate_from_bool_list(state["clean_sheet_last5"]),
        "hist_failed_to_score_rate_last5": _rate_from_bool_list(state["failed_to_score_last5"]),
        "hist_days_since_last_match": days_since_last,
        "hist_has_history": float(state["matches_played"] > 0),
    }

def snapshot_h2h_features(state: dict) -> dict:
    return {
        "h2h_matches_played_pre": float(state["matches_played"]),
        "h2h_points_a_avg_last3": _mean_or_nan(state["points_a_last3"]),
        "h2h_gd_a_avg_last3": _mean_or_nan(state["gd_a_last3"]),
        "h2h_total_goals_avg_last3": _mean_or_nan(state["total_last3"]),
        "h2h_has_history": float(state["matches_played"] > 0),
    }

def update_states_from_score(
    state_a: dict,
    state_b: dict,
    h2h_state: dict,
    match_context: dict,
    goals_a: int,
    goals_b: int,
) -> tuple[dict, dict, dict]:
    state_a = copy.deepcopy(state_a)
    state_b = copy.deepcopy(state_b)
    h2h_state = copy.deepcopy(h2h_state)

    goals_a = int(goals_a)
    goals_b = int(goals_b)
    gd = goals_a - goals_b
    total = goals_a + goals_b

    points_a = 3 if gd > 0 else 1 if gd == 0 else 0
    points_b = 3 if gd < 0 else 1 if gd == 0 else 0

    home_side = int(match_context.get("home_side", 0))
    tournament_weight = float(match_context.get("tournament_weight", 1.20))

    home_bonus = 60.0 if home_side == 1 else -60.0 if home_side == -1 else 0.0
    expected_a = expected_elo_result(state_a["elo_overall"], state_b["elo_overall"], home_bonus=home_bonus)
    actual_a = 1.0 if gd > 0 else 0.5 if gd == 0 else 0.0

    k_elo = 24.0
    delta_elo = k_elo * tournament_weight * (actual_a - expected_a)
    state_a["elo_overall"] += delta_elo
    state_b["elo_overall"] -= delta_elo

    gd_home_bonus = 10.0 if home_side == 1 else -10.0 if home_side == -1 else 0.0
    expected_gd = (state_a["elo_goal_diff"] + gd_home_bonus - state_b["elo_goal_diff"]) / 100.0
    resid = gd - expected_gd
    k_gd = 6.0
    state_a["elo_goal_diff"] += k_gd * resid
    state_b["elo_goal_diff"] -= k_gd * resid

    alpha = 0.35
    for state, pts, gf, ga, gd_side in [
        (state_a, points_a, goals_a, goals_b, gd),
        (state_b, points_b, goals_b, goals_a, -gd),
    ]:
        state["ewm_points"] = alpha * pts + (1.0 - alpha) * state["ewm_points"]
        state["ewm_gf"] = alpha * gf + (1.0 - alpha) * state["ewm_gf"]
        state["ewm_ga"] = alpha * ga + (1.0 - alpha) * state["ewm_ga"]
        state["ewm_gd"] = alpha * gd_side + (1.0 - alpha) * state["ewm_gd"]

        state["points_last5"].append(pts)
        state["points_last10"].append(pts)
        state["gf_last5"].append(gf)
        state["ga_last5"].append(ga)
        state["gd_last5"].append(gd_side)
        state["result_last10"].append(int(np.sign(gd_side)))
        state["clean_sheet_last5"].append(int(ga == 0))
        state["failed_to_score_last5"].append(int(gf == 0))
        state["matches_played"] += 1
        state["last_match_date"] = match_context["date"]

    h2h_state["matches_played"] += 1
    h2h_state["points_a_last3"].append(points_a)
    h2h_state["gd_a_last3"].append(gd)
    h2h_state["total_last3"].append(total)

    return state_a, state_b, h2h_state


In [ ]:
def build_train_history_features_full(train_match_df: pd.DataFrame) -> tuple[pd.DataFrame, dict, dict]:
    df = train_match_df.sort_values(["date", "match_id"]).reset_index(drop=True).copy()

    team_states = {}
    h2h_states = {}
    rows = []

    for row in tqdm(df.itertuples(index=False), total=len(df), desc="build_train_history_full"):
        gender = str(row.gender)
        team_a = str(row.team_a)
        team_b = str(row.team_b)
        key_a = (gender, team_a)
        key_b = (gender, team_b)
        h2h_key = (gender, team_a, team_b)

        if key_a not in team_states:
            team_states[key_a] = init_team_state()
        if key_b not in team_states:
            team_states[key_b] = init_team_state()
        if h2h_key not in h2h_states:
            h2h_states[h2h_key] = init_h2h_state()

        feat_a = snapshot_team_features_full(team_states[key_a], row.date)
        feat_b = snapshot_team_features_full(team_states[key_b], row.date)
        feat_h2h = snapshot_h2h_features(h2h_states[h2h_key])

        out = {"match_id": row.match_id}
        for k, v in feat_a.items():
            out[f"{k}_a"] = v
        for k, v in feat_b.items():
            out[f"{k}_b"] = v
        out.update(feat_h2h)

        diff_pairs = [
            "hist_matches_played", "hist_elo_overall", "hist_elo_gd", "hist_ewm_points", "hist_ewm_gf",
            "hist_ewm_ga", "hist_ewm_gd", "hist_points_avg_last5", "hist_points_avg_last10",
            "hist_gf_avg_last5", "hist_ga_avg_last5", "hist_gd_avg_last5", "hist_win_rate_last10",
            "hist_draw_rate_last10", "hist_loss_rate_last10", "hist_clean_sheet_rate_last5",
            "hist_failed_to_score_rate_last5", "hist_days_since_last_match", "hist_has_history",
        ]
        for stem in diff_pairs:
            a_val = out.get(f"{stem}_a", np.nan)
            b_val = out.get(f"{stem}_b", np.nan)
            out[f"{stem}_diff"] = a_val - b_val if pd.notna(a_val) and pd.notna(b_val) else np.nan

        rows.append(out)

        match_context = {
            "home_side": int(row.home_side) if hasattr(row, "home_side") else 0,
            "tournament_weight": get_tournament_weight(row.tournament),
            "date": row.date,
        }
        updated_a, updated_b, updated_h2h = update_states_from_score(
            team_states[key_a], team_states[key_b], h2h_states[h2h_key],
            match_context, row.team_a_goals, row.team_b_goals
        )
        team_states[key_a] = updated_a
        team_states[key_b] = updated_b
        h2h_states[h2h_key] = updated_h2h

    hist_df = pd.DataFrame(rows)
    return hist_df, team_states, h2h_states

def build_future_history_features_freeze(
    future_match_df: pd.DataFrame,
    team_states_at_cutoff: dict,
    h2h_states_at_cutoff: dict,
) -> pd.DataFrame:
    df = future_match_df.sort_values(["date", "match_id"]).reset_index(drop=True).copy()
    team_states = copy.deepcopy(team_states_at_cutoff)
    h2h_states = copy.deepcopy(h2h_states_at_cutoff)

    rows = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="build_future_history_freeze"):
        gender = str(row.gender)
        team_a = str(row.team_a)
        team_b = str(row.team_b)
        key_a = (gender, team_a)
        key_b = (gender, team_b)
        h2h_key = (gender, team_a, team_b)

        if key_a not in team_states:
            team_states[key_a] = init_team_state()
        if key_b not in team_states:
            team_states[key_b] = init_team_state()
        if h2h_key not in h2h_states:
            h2h_states[h2h_key] = init_h2h_state()

        feat_a = snapshot_team_features_full(team_states[key_a], row.date)
        feat_b = snapshot_team_features_full(team_states[key_b], row.date)
        feat_h2h = snapshot_h2h_features(h2h_states[h2h_key])

        out = {"match_id": row.match_id}
        for k, v in feat_a.items():
            out[f"{k}_a"] = v
        for k, v in feat_b.items():
            out[f"{k}_b"] = v
        out.update(feat_h2h)

        diff_pairs = [
            "hist_matches_played", "hist_elo_overall", "hist_elo_gd", "hist_ewm_points", "hist_ewm_gf",
            "hist_ewm_ga", "hist_ewm_gd", "hist_points_avg_last5", "hist_points_avg_last10",
            "hist_gf_avg_last5", "hist_ga_avg_last5", "hist_gd_avg_last5", "hist_win_rate_last10",
            "hist_draw_rate_last10", "hist_loss_rate_last10", "hist_clean_sheet_rate_last5",
            "hist_failed_to_score_rate_last5", "hist_days_since_last_match", "hist_has_history",
        ]
        for stem in diff_pairs:
            a_val = out.get(f"{stem}_a", np.nan)
            b_val = out.get(f"{stem}_b", np.nan)
            out[f"{stem}_diff"] = a_val - b_val if pd.notna(a_val) and pd.notna(b_val) else np.nan

        rows.append(out)

        team_states[key_a]["last_match_date"] = row.date
        team_states[key_b]["last_match_date"] = row.date

    return pd.DataFrame(rows)


In [ ]:
train_static_features = engineer_static_match_features(train_match_base)
test_static_features = engineer_static_match_features(test_match_base)

train_history_full, final_team_states_after_train, final_h2h_states_after_train = build_train_history_features_full(train_static_features)

train_student_features = train_static_features.merge(train_history_full, on="match_id", how="left")

print("[INFO] train_student_features shape:", train_student_features.shape)
display(train_student_features.head(3))


## 06. Teacher Feature Construction

Teacher memakai semua fitur student, lalu ditambah **teacher-only official train features** yang memang tidak tersedia di test.  
Bagian ini penting karena teacher dipakai untuk mengukur headroom dan menghasilkan signal distillation, tetapi **teacher tidak boleh dipakai langsung untuk submission final**.


In [ ]:
def identify_teacher_only_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> list[str]:
    row_level_exclude = {
        "Id", "match_id", "date", "team", "opponent", "gender", "tournament", "venue_country", "neutral",
        "team_goals", "opp_goals",
    }
    teacher_only = [c for c in train_df.columns if (c not in test_df.columns) and (c not in row_level_exclude)]
    return sorted(teacher_only)

teacher_only_feature_cols = identify_teacher_only_columns(train, test)
print("[INFO] jumlah teacher-only row-level cols:", len(teacher_only_feature_cols))
print("[INFO] contoh teacher-only cols:", teacher_only_feature_cols[:30])


In [ ]:
def build_teacher_feature_set(train_match_df: pd.DataFrame) -> pd.DataFrame:
    """Bangun match-level teacher-only feature block dari train match data."""
    match_level_teacher_cols = []
    for col in teacher_only_feature_cols:
        a_col = f"{col}_a"
        b_col = f"{col}_b"
        if a_col in train_match_df.columns:
            match_level_teacher_cols.append(a_col)
        if b_col in train_match_df.columns:
            match_level_teacher_cols.append(b_col)

    teacher_only_df = train_match_df[["match_id"] + match_level_teacher_cols].copy()
    return teacher_only_df

train_teacher_only_features = build_teacher_feature_set(train_match_base)
train_teacher_features = train_student_features.merge(train_teacher_only_features, on="match_id", how="left")

print("[INFO] train_teacher_only_features shape:", train_teacher_only_features.shape)
print("[INFO] train_teacher_features shape     :", train_teacher_features.shape)
display(train_teacher_features.head(3))


## 07. Temporal Holdout dan Scoreline Prior

Split validasi tetap dilakukan di level match secara temporal.  
Semua evaluasi utama di notebook ini memakai split yang sama, supaya comparison tetap apple-to-apple.


In [ ]:
train_fold_base, valid_fold_base = make_time_based_holdout(train_student_features, valid_fraction=0.2)

print("[INFO] train_fold_base shape:", train_fold_base.shape)
print("[INFO] valid_fold_base shape:", valid_fold_base.shape)
print("[INFO] train_fold date range:", train_fold_base["date"].min(), "->", train_fold_base["date"].max())
print("[INFO] valid_fold date range:", valid_fold_base["date"].min(), "->", valid_fold_base["date"].max())
print("[INFO] overlap match_id:", len(set(train_fold_base["match_id"]).intersection(set(valid_fold_base["match_id"]))))


In [ ]:
# Build valid/test freeze history from cutoff states learned only on train_fold.
train_fold_for_state = train_static_features[train_static_features["match_id"].isin(train_fold_base["match_id"])].copy()
valid_fold_static = train_static_features[train_static_features["match_id"].isin(valid_fold_base["match_id"])].copy()

train_fold_history_full, cutoff_team_states_valid, cutoff_h2h_states_valid = build_train_history_features_full(train_fold_for_state)
valid_history_full_freeze = build_future_history_features_freeze(
    valid_fold_static,
    cutoff_team_states_valid,
    cutoff_h2h_states_valid,
)

train_fold_student_df = train_fold_for_state.merge(train_fold_history_full, on="match_id", how="left")
valid_fold_student_df = valid_fold_static.merge(valid_history_full_freeze, on="match_id", how="left")

train_fold_teacher_only = train_teacher_only_features[train_teacher_only_features["match_id"].isin(train_fold_base["match_id"])].copy()
valid_fold_teacher_only = train_teacher_only_features[train_teacher_only_features["match_id"].isin(valid_fold_base["match_id"])].copy()

train_fold_teacher_df = train_fold_student_df.merge(train_fold_teacher_only, on="match_id", how="left")
valid_fold_teacher_df = valid_fold_student_df.merge(valid_fold_teacher_only, on="match_id", how="left")

print("[INFO] train_fold_student_df shape:", train_fold_student_df.shape)
print("[INFO] valid_fold_student_df shape:", valid_fold_student_df.shape)
print("[INFO] train_fold_teacher_df shape:", train_fold_teacher_df.shape)
print("[INFO] valid_fold_teacher_df shape:", valid_fold_teacher_df.shape)


In [ ]:
def build_scoreline_prior(goal_a: pd.Series, goal_b: pd.Series, max_goals: int, alpha: float = 1.0) -> dict:
    clipped_a = goal_a.clip(0, max_goals).astype(int)
    clipped_b = goal_b.clip(0, max_goals).astype(int)

    counter = Counter(zip(clipped_a, clipped_b))
    all_pairs = [(a, b) for a in range(max_goals + 1) for b in range(max_goals + 1)]
    smoothed = {pair: counter.get(pair, 0) + alpha for pair in all_pairs}
    total = float(sum(smoothed.values()))
    return {pair: val / total for pair, val in smoothed.items()}

scoreline_prior_lookup = {
    mg: build_scoreline_prior(train_fold_base["team_a_goals"], train_fold_base["team_b_goals"], max_goals=mg, alpha=1.0)
    for mg in [7, 9]
}
print("[OK] scoreline prior lookup siap.")


## 08. Target Construction

Di tahap ini kita membentuk:

- target utama backbone deterministic,
- dan target **directional tail** yang akan dipakai baik oleh teacher maupun oleh student tail-aware branch.


In [ ]:
y_train_fold = train_fold_base[["match_id", "team_a_goals", "team_b_goals", "tournament", "gender", "neutral"]].copy()
y_train_fold["y_goal_a"] = y_train_fold["team_a_goals"]
y_train_fold["y_goal_b"] = y_train_fold["team_b_goals"]
y_train_fold["y_total"] = y_train_fold["team_a_goals"] + y_train_fold["team_b_goals"]
y_train_fold["y_gd"] = y_train_fold["team_a_goals"] - y_train_fold["team_b_goals"]
y_train_fold["y_outcome"] = [
    _outcome(a, b) for a, b in zip(y_train_fold["team_a_goals"], y_train_fold["team_b_goals"])
]

y_valid = valid_fold_base[["match_id", "team_a_goals", "team_b_goals", "tournament", "gender", "neutral"]].copy()
y_valid["y_goal_a"] = y_valid["team_a_goals"]
y_valid["y_goal_b"] = y_valid["team_b_goals"]
y_valid["y_total"] = y_valid["team_a_goals"] + y_valid["team_b_goals"]
y_valid["y_gd"] = y_valid["team_a_goals"] - y_valid["team_b_goals"]
y_valid["y_outcome"] = [
    _outcome(a, b) for a, b in zip(y_valid["team_a_goals"], y_valid["team_b_goals"])
]

for target_df in [y_train_fold, y_valid]:
    target_df["is_team_a_blowout_5plus"] = (target_df["y_gd"] >= 5).astype(int)
    target_df["is_team_b_blowout_5plus"] = (target_df["y_gd"] <= -5).astype(int)
    target_df["is_team_a_blowout_7plus"] = (target_df["y_gd"] >= 7).astype(int)
    target_df["is_team_b_blowout_7plus"] = (target_df["y_gd"] <= -7).astype(int)
    target_df["is_high_total"] = (target_df["y_total"] >= 6).astype(int)

directional_tail_target_definitions = {
    "is_team_a_blowout_5plus": "1 jika goal_diff >= 5",
    "is_team_b_blowout_5plus": "1 jika goal_diff <= -5",
    "is_team_a_blowout_7plus": "1 jika goal_diff >= 7",
    "is_team_b_blowout_7plus": "1 jika goal_diff <= -7",
    "is_high_total": "1 jika total_goals >= 6",
}
with open(SUM_DIR / "directional_tail_target_definitions.json", "w", encoding="utf-8") as f:
    json.dump(directional_tail_target_definitions, f, indent=2, ensure_ascii=False)

display(y_train_fold.head())


## 09. Student Anchor Baseline

Variant ini adalah baseline submission-safe utama di notebook ini.  
Student anchor hanya memakai fitur legal, dilatih pada hard labels asli, lalu dievaluasi dengan decoder deterministic backbone.


In [ ]:
categorical_features = [
    "team_a", "team_b", "gender", "tournament", "venue_country",
    "team_a_confederation", "team_b_confederation",
    "pair_key", "confed_pair_key",
]
student_feature_cols = [
    c for c in train_fold_student_df.columns
    if c not in {
        "match_id", "id_a", "id_b", "date",
        "team_a_goals", "team_b_goals",
    }
]

student_cat_features = [c for c in categorical_features if c in student_feature_cols]
student_num_features = [c for c in student_feature_cols if c not in student_cat_features]

teacher_only_match_cols = [
    c for c in train_fold_teacher_df.columns
    if c not in train_fold_student_df.columns and c not in {"match_id"}
]
teacher_feature_cols = student_feature_cols + teacher_only_match_cols

with open(SUM_DIR / "student_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(student_feature_cols, f, indent=2, ensure_ascii=False)
with open(SUM_DIR / "teacher_only_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(teacher_only_match_cols, f, indent=2, ensure_ascii=False)
with open(SUM_DIR / "teacher_feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(teacher_feature_cols, f, indent=2, ensure_ascii=False)

print("[INFO] jumlah student features    :", len(student_feature_cols))
print("[INFO] jumlah teacher-only match  :", len(teacher_only_match_cols))
print("[INFO] jumlah teacher features    :", len(teacher_feature_cols))


In [ ]:
SAFE_GOAL_OBJECTIVE_NAME = "tweedie"

goal_reg_params_map = {
    "rmse": dict(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=200,
    ),
    "poisson": dict(
        loss_function="Poisson",
        eval_metric="Poisson",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=200,
    ),
    "tweedie": dict(
        loss_function="Tweedie:variance_power=1.3",
        eval_metric="Tweedie:variance_power=1.3",
        iterations=1500,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        random_seed=SEED,
        allow_writing_files=False,
        verbose=200,
    ),
}

goal_reg_params = goal_reg_params_map[SAFE_GOAL_OBJECTIVE_NAME]
total_reg_params = goal_reg_params_map["rmse"]
gd_reg_params = goal_reg_params_map["rmse"]

outcome_clf_params = dict(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=1500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
)

tail_clf_params = dict(
    loss_function="Logloss",
    eval_metric="Logloss",
    iterations=1200,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=200,
)


In [ ]:
def make_inner_temporal_split(df: pd.DataFrame, eval_fraction: float = 0.15) -> tuple[np.ndarray, np.ndarray]:
    order = np.arange(len(df))
    n_eval = max(1, int(len(df) * eval_fraction))
    split_idx = len(df) - n_eval
    tr_idx = order[:split_idx]
    ev_idx = order[split_idx:]
    return tr_idx, ev_idx

def _align_feature_columns(feature_df: pd.DataFrame, feature_cols: list[str], cat_features: list[str] | None = None, context: str = ""):
    missing_cols = [c for c in feature_cols if c not in feature_df.columns]
    if missing_cols:
        ctx = f" ({context})" if context else ""
        print(f"[WARN] feature alignment{ctx}: dropping {len(missing_cols)} missing columns")
        print(f"[WARN] sample missing: {missing_cols[:12]}")

    aligned_feature_cols = [c for c in feature_cols if c in feature_df.columns]
    if len(aligned_feature_cols) == 0:
        raise ValueError("Tidak ada feature valid setelah alignment terhadap dataframe input.")

    if cat_features is None:
        return aligned_feature_cols, []

    aligned_cat_features = [c for c in cat_features if c in aligned_feature_cols]
    return aligned_feature_cols, aligned_cat_features

def _build_prediction_feature_frame(feature_df: pd.DataFrame, trained_models: dict, context: str = "") -> pd.DataFrame:
    """Rebuild feature frame in exact training order to keep CatBoost schema consistent."""
    feature_cols = trained_models["feature_cols"]
    cat_cols = trained_models.get("cat_features", [c for c in feature_cols if c in categorical_features])

    missing_cols = [c for c in feature_cols if c not in feature_df.columns]
    if missing_cols:
        ctx = f" ({context})" if context else ""
        print(f"[WARN] prediction schema{ctx}: imputing {len(missing_cols)} missing columns")
        print(f"[WARN] sample missing: {missing_cols[:12]}")

    X = pd.DataFrame(index=feature_df.index)
    for col in feature_cols:
        if col in feature_df.columns:
            X[col] = feature_df[col]
        else:
            X[col] = "__MISSING__" if col in cat_cols else np.nan

    for col in cat_cols:
        if col in X.columns:
            X[col] = X[col].astype("string").fillna("__MISSING__").astype(str)

    return X

def fit_regressor(X: pd.DataFrame, y: pd.Series, cat_features: list[str], params: dict):
    tr_idx, ev_idx = make_inner_temporal_split(X, eval_fraction=0.15)
    model = CatBoostRegressor(**params)
    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        cat_features=cat_features,
        eval_set=(X.iloc[ev_idx], y.iloc[ev_idx]),
        use_best_model=True,
        verbose=params.get("verbose", 200),
    )
    return model

def fit_classifier(X: pd.DataFrame, y: pd.Series, cat_features: list[str], params: dict):
    tr_idx, ev_idx = make_inner_temporal_split(X, eval_fraction=0.15)
    model = CatBoostClassifier(**params)
    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        cat_features=cat_features,
        eval_set=(X.iloc[ev_idx], y.iloc[ev_idx]),
        use_best_model=True,
        verbose=params.get("verbose", 200),
    )
    return model

def predict_raw_outputs_for_df(feature_df: pd.DataFrame, trained_models: dict) -> pd.DataFrame:
    raw = pd.DataFrame({
        "match_id": feature_df["match_id"].values,
        "tournament": feature_df["tournament"].values,
    })

    if "team_a_goals" in feature_df.columns and "team_b_goals" in feature_df.columns:
        raw["actual_team_a_goals"] = feature_df["team_a_goals"].values
        raw["actual_team_b_goals"] = feature_df["team_b_goals"].values

    X = _build_prediction_feature_frame(feature_df, trained_models, context="predict_raw_outputs_for_df")

    raw["pred_goal_a_cont"] = trained_models["goal_a"].predict(X)
    raw["pred_goal_b_cont"] = trained_models["goal_b"].predict(X)
    raw["pred_total_cont"] = trained_models["total"].predict(X)
    raw["pred_gd_cont"] = trained_models["gd"].predict(X)

    outcome_proba = trained_models["outcome"].predict_proba(X)
    raw["pred_outcome_proba_0"] = outcome_proba[:, 0]
    raw["pred_outcome_proba_1"] = outcome_proba[:, 1]
    raw["pred_outcome_proba_2"] = outcome_proba[:, 2]
    return raw

def fit_hard_model_block(feature_df: pd.DataFrame, feature_cols: list[str], cat_features: list[str], target_df: pd.DataFrame) -> dict:
    aligned_feature_cols, aligned_cat_features = _align_feature_columns(
        feature_df=feature_df,
        feature_cols=feature_cols,
        cat_features=cat_features,
        context="fit_hard_model_block",
    )
    X = feature_df[aligned_feature_cols].copy()

    models = {
        "feature_cols": aligned_feature_cols,
        "cat_features": aligned_cat_features,
        "goal_a": fit_regressor(X, target_df["y_goal_a"], aligned_cat_features, goal_reg_params),
        "goal_b": fit_regressor(X, target_df["y_goal_b"], aligned_cat_features, goal_reg_params),
        "total": fit_regressor(X, target_df["y_total"], aligned_cat_features, total_reg_params),
        "gd": fit_regressor(X, target_df["y_gd"], aligned_cat_features, gd_reg_params),
        "outcome": fit_classifier(X, target_df["y_outcome"], aligned_cat_features, outcome_clf_params),
    }
    return models

In [ ]:
def _build_anchor_candidate_grid(max_goals: int, scoreline_prior: dict, eps: float = 1e-12) -> dict:
    a_vals = np.arange(max_goals + 1, dtype=np.int16)
    b_vals = np.arange(max_goals + 1, dtype=np.int16)
    cand_a, cand_b = np.meshgrid(a_vals, b_vals, indexing="ij")
    cand_a = cand_a.reshape(-1)
    cand_b = cand_b.reshape(-1)
    cand_total = cand_a + cand_b
    cand_gd = cand_a - cand_b
    cand_outcome = np.where(cand_a > cand_b, 0, np.where(cand_a == cand_b, 1, 2)).astype(np.int8)
    prior_vals = np.array([float(scoreline_prior.get((int(a), int(b)), eps)) for a, b in zip(cand_a, cand_b)], dtype=np.float64)
    prior_vals = np.clip(prior_vals, eps, 1.0)
    return {
        "cand_a": cand_a,
        "cand_b": cand_b,
        "cand_total": cand_total,
        "cand_gd": cand_gd,
        "cand_outcome": cand_outcome,
        "prior_vals": prior_vals,
    }

def decode_anchor_batch_from_raw(raw_df: pd.DataFrame, decoder_params: dict, scoreline_prior: dict, eps: float = 1e-12) -> pd.DataFrame:
    max_goals = int(decoder_params["MAX_GOALS"])
    grid = _build_anchor_candidate_grid(max_goals=max_goals, scoreline_prior=scoreline_prior, eps=eps)

    cand_a = grid["cand_a"][None, :].astype(np.float64)
    cand_b = grid["cand_b"][None, :].astype(np.float64)
    cand_total = grid["cand_total"][None, :].astype(np.float64)
    cand_gd = grid["cand_gd"][None, :].astype(np.float64)
    cand_outcome = grid["cand_outcome"]
    prior_cost = -np.log(grid["prior_vals"] + eps)[None, :]

    pred_goal_a = raw_df["pred_goal_a_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_goal_b = raw_df["pred_goal_b_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_total = raw_df["pred_total_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_gd = raw_df["pred_gd_cont"].to_numpy(dtype=np.float64)[:, None]

    outcome_probs = raw_df[
        ["pred_outcome_proba_0", "pred_outcome_proba_1", "pred_outcome_proba_2"]
    ].to_numpy(dtype=np.float64)
    outcome_probs = np.clip(outcome_probs, eps, 1.0)
    outcome_cost = -np.log(outcome_probs[:, cand_outcome] + eps)

    total_cost = (
        float(decoder_params["w_direct"]) * (np.abs(cand_a - pred_goal_a) + np.abs(cand_b - pred_goal_b))
        + float(decoder_params["w_total"]) * np.abs(cand_total - pred_total)
        + float(decoder_params["w_gd"]) * np.abs(cand_gd - pred_gd)
        + float(decoder_params["w_outcome"]) * outcome_cost
        + float(decoder_params["w_prior"]) * prior_cost
    )
    best_idx = np.argmin(total_cost, axis=1)

    return pd.DataFrame(
        {
            "match_id": raw_df["match_id"].values,
            "pred_team_a_goals": grid["cand_a"][best_idx].astype(int),
            "pred_team_b_goals": grid["cand_b"][best_idx].astype(int),
        }
    )

def tune_decoder_from_raw_outputs(raw_df: pd.DataFrame, decoder_grid: dict, scoreline_prior: dict) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    keys = list(decoder_grid.keys())
    combos = list(product(*[decoder_grid[k] for k in keys]))
    for combo in tqdm(combos, total=len(combos), desc="tune_anchor_decoder"):
        params = dict(zip(keys, combo))
        prior = scoreline_prior[int(params["MAX_GOALS"])]
        pred_df = decode_anchor_batch_from_raw(raw_df, params, prior)
        score = awmae_score(
            raw_df["actual_team_a_goals"],
            raw_df["actual_team_b_goals"],
            pred_df["pred_team_a_goals"],
            pred_df["pred_team_b_goals"],
            raw_df["tournament"],
        )
        rows.append({**params, "valid_awmae": float(score)})
    grid_df = pd.DataFrame(rows).sort_values("valid_awmae").reset_index(drop=True)
    best = grid_df.iloc[0].to_dict()
    pred_df = decode_anchor_batch_from_raw(raw_df, best, scoreline_prior[int(best["MAX_GOALS"])])
    return grid_df, best, pred_df


In [ ]:
anchor_decoder_grid = {
    "MAX_GOALS": [7, 9],
    "w_direct": [0.75, 1.0],
    "w_total": [1.0],
    "w_gd": [1.0],
    "w_outcome": [1.5, 2.0],
    "w_prior": [0.2, 0.5],
}

student_anchor_models = fit_hard_model_block(
    train_fold_student_df,
    feature_cols=student_feature_cols,
    cat_features=student_cat_features,
    target_df=y_train_fold,
)

student_anchor_raw_valid = predict_raw_outputs_for_df(valid_fold_student_df, student_anchor_models)
decoder_grid_anchor, best_decoder_anchor, student_anchor_pred_valid = tune_decoder_from_raw_outputs(
    student_anchor_raw_valid,
    anchor_decoder_grid,
    scoreline_prior_lookup,
)

valid_pred_match_student_anchor = student_anchor_raw_valid.merge(student_anchor_pred_valid, on="match_id", how="left")
valid_pred_match_student_anchor.to_csv(PRED_DIR / "valid_pred_match_student_anchor.csv", index=False)
decoder_grid_anchor.to_csv(SUM_DIR / "decoder_grid_anchor.csv", index=False)

awmae_student_anchor = awmae_score(
    valid_pred_match_student_anchor["actual_team_a_goals"],
    valid_pred_match_student_anchor["actual_team_b_goals"],
    valid_pred_match_student_anchor["pred_team_a_goals"],
    valid_pred_match_student_anchor["pred_team_b_goals"],
    valid_pred_match_student_anchor["tournament"],
)

print(f"[RESULT] student_anchor valid AW-MAE = {awmae_student_anchor:.6f}")
display(pd.DataFrame([best_decoder_anchor]))


## 10. Teacher Upper Bound

Teacher memakai fitur student + fitur train-only official.  
Variant ini hanya dipakai untuk **analysis** dan **headroom measurement**, bukan untuk submission-safe pipeline.


In [ ]:
def predict_teacher_signals_for_valid(teacher_models: dict, valid_teacher_df: pd.DataFrame) -> pd.DataFrame:
    X = _build_prediction_feature_frame(valid_teacher_df, teacher_models, context="predict_teacher_signals_for_valid")
    out = pd.DataFrame({"match_id": valid_teacher_df["match_id"].values})

    out["teacher_goal_a_hat"] = teacher_models["goal_a"].predict(X)
    out["teacher_goal_b_hat"] = teacher_models["goal_b"].predict(X)
    out["teacher_total_hat"] = teacher_models["total"].predict(X)
    out["teacher_gd_hat"] = teacher_models["gd"].predict(X)

    proba = teacher_models["outcome"].predict_proba(X)
    out["teacher_p_a_win"] = proba[:, 0]
    out["teacher_p_draw"] = proba[:, 1]
    out["teacher_p_b_win"] = proba[:, 2]

    out["teacher_p_a_blowout_5plus"] = teacher_models["a_blowout_5"].predict_proba(X)[:, 1]
    out["teacher_p_b_blowout_5plus"] = teacher_models["b_blowout_5"].predict_proba(X)[:, 1]
    out["teacher_p_a_blowout_7plus"] = teacher_models["a_blowout_7"].predict_proba(X)[:, 1]
    out["teacher_p_b_blowout_7plus"] = teacher_models["b_blowout_7"].predict_proba(X)[:, 1]
    out["teacher_p_high_total"] = teacher_models["high_total"].predict_proba(X)[:, 1]
    return out

def fit_teacher_models(feature_df: pd.DataFrame, feature_cols: list[str], cat_features: list[str], target_df: pd.DataFrame) -> dict:
    # Guard against stale feature lists from previous notebook runs/schemas.
    missing_cols = [c for c in feature_cols if c not in feature_df.columns]
    if missing_cols:
        print(f"[WARN] fit_teacher_models dropping missing feature cols: {len(missing_cols)}")
        print(f"[WARN] sample missing: {missing_cols[:12]}")

    feature_cols = [c for c in feature_cols if c in feature_df.columns]
    cat_features = [c for c in cat_features if c in feature_cols]

    if len(feature_cols) == 0:
        raise ValueError("Tidak ada teacher feature yang valid setelah sinkronisasi kolom.")

    X = feature_df[feature_cols].copy()
    models = {
        "feature_cols": feature_cols,
        'cat_features': cat_features,
        "goal_a": fit_regressor(X, target_df["y_goal_a"], cat_features, goal_reg_params),
        "goal_b": fit_regressor(X, target_df["y_goal_b"], cat_features, goal_reg_params),
        "total": fit_regressor(X, target_df["y_total"], cat_features, total_reg_params),
        "gd": fit_regressor(X, target_df["y_gd"], cat_features, gd_reg_params),
        "outcome": fit_classifier(X, target_df["y_outcome"], cat_features, outcome_clf_params),
        "a_blowout_5": fit_classifier(X, target_df["is_team_a_blowout_5plus"], cat_features, tail_clf_params),
        "b_blowout_5": fit_classifier(X, target_df["is_team_b_blowout_5plus"], cat_features, tail_clf_params),
        "a_blowout_7": fit_classifier(X, target_df["is_team_a_blowout_7plus"], cat_features, tail_clf_params),
        "b_blowout_7": fit_classifier(X, target_df["is_team_b_blowout_7plus"], cat_features, tail_clf_params),
        "high_total": fit_classifier(X, target_df["is_high_total"], cat_features, tail_clf_params),
    }
    return models

# Rebuild teacher feature list from CURRENT frames to avoid stale runtime variables.
teacher_only_match_cols = [
    c for c in train_fold_teacher_df.columns
    if c not in train_fold_student_df.columns and c not in {"match_id"}
]
teacher_feature_cols = student_feature_cols + teacher_only_match_cols
teacher_feature_cols = [c for c in teacher_feature_cols if c in train_fold_teacher_df.columns]
teacher_cat_features = [c for c in teacher_feature_cols if c in categorical_features]

teacher_models = fit_teacher_models(
    train_fold_teacher_df,
    feature_cols=teacher_feature_cols,
    cat_features=teacher_cat_features,
    target_df=y_train_fold,
)

teacher_valid_signals = predict_teacher_signals_for_valid(teacher_models, valid_fold_teacher_df)

teacher_raw_valid = pd.DataFrame({
    "match_id": valid_fold_teacher_df["match_id"].values,
    "tournament": valid_fold_teacher_df["tournament"].values,
    "actual_team_a_goals": valid_fold_teacher_df["team_a_goals"].values,
    "actual_team_b_goals": valid_fold_teacher_df["team_b_goals"].values,
    "pred_goal_a_cont": teacher_valid_signals["teacher_goal_a_hat"].values,
    "pred_goal_b_cont": teacher_valid_signals["teacher_goal_b_hat"].values,
    "pred_total_cont": teacher_valid_signals["teacher_total_hat"].values,
    "pred_gd_cont": teacher_valid_signals["teacher_gd_hat"].values,
    "pred_outcome_proba_0": teacher_valid_signals["teacher_p_a_win"].values,
    "pred_outcome_proba_1": teacher_valid_signals["teacher_p_draw"].values,
    "pred_outcome_proba_2": teacher_valid_signals["teacher_p_b_win"].values,
})

teacher_decoder_grid, teacher_best_decoder, teacher_pred_valid = tune_decoder_from_raw_outputs(
    teacher_raw_valid,
    anchor_decoder_grid,
    scoreline_prior_lookup,
)

valid_pred_match_teacher_upper_bound = teacher_raw_valid.merge(teacher_pred_valid, on="match_id", how="left")
valid_pred_match_teacher_upper_bound.to_csv(PRED_DIR / "valid_pred_match_teacher_upper_bound.csv", index=False)

awmae_teacher_upper_bound = awmae_score(
    valid_pred_match_teacher_upper_bound["actual_team_a_goals"],
    valid_pred_match_teacher_upper_bound["actual_team_b_goals"],
    valid_pred_match_teacher_upper_bound["pred_team_a_goals"],
    valid_pred_match_teacher_upper_bound["pred_team_b_goals"],
    valid_pred_match_teacher_upper_bound["tournament"],
)

print(f"[RESULT] teacher_upper_bound valid AW-MAE = {awmae_teacher_upper_bound:.6f}")

## 11. Build OOF Teacher Signals

Teacher signal untuk distillation train **harus fair**.  
Karena itu kita buat OOF-like teacher signal dengan **temporal blocks** kecil. Default di notebook ini adalah **3 blocks**, dan secara guardrail maksimal **4**.


In [ ]:
teacher_signal_columns = [
    "teacher_p_a_win",
    "teacher_p_draw",
    "teacher_p_b_win",
    "teacher_goal_a_hat",
    "teacher_goal_b_hat",
    "teacher_total_hat",
    "teacher_gd_hat",
    "teacher_p_a_blowout_5plus",
    "teacher_p_b_blowout_5plus",
    "teacher_p_a_blowout_7plus",
    "teacher_p_b_blowout_7plus",
    "teacher_p_high_total",
]
with open(SUM_DIR / "teacher_signal_columns.json", "w", encoding="utf-8") as f:
    json.dump(teacher_signal_columns, f, indent=2, ensure_ascii=False)


In [ ]:
def build_teacher_oof_signals(
    train_fold_teacher_df: pd.DataFrame,
    train_fold_targets: pd.DataFrame,
    n_temporal_blocks: int = 3,
) -> pd.DataFrame:
    if n_temporal_blocks > 4:
        raise ValueError("n_temporal_blocks tidak boleh lebih dari 4 di EXP04B.")

    df = train_fold_teacher_df.sort_values(["date", "match_id"]).reset_index(drop=True).copy()
    tgt = train_fold_targets.set_index("match_id").loc[df["match_id"]].reset_index()

    warmup_size = max(300, int(len(df) * 0.25))
    remain_idx = np.arange(warmup_size, len(df))
    block_splits = np.array_split(remain_idx, n_temporal_blocks)

    oof_rows = []
    for block_id, pred_idx in enumerate(block_splits, start=1):
        pred_idx = np.asarray(pred_idx, dtype=int)
        if len(pred_idx) == 0:
            continue
        train_idx = np.arange(0, pred_idx.min())
        if len(train_idx) < 200:
            continue

        X_train = df.iloc[train_idx].reset_index(drop=True)
        y_train = tgt.iloc[train_idx].reset_index(drop=True)
        X_pred = df.iloc[pred_idx].reset_index(drop=True)

        print(f"[INFO] teacher OOF block {block_id}: train={len(X_train)} | pred={len(X_pred)}")
        block_models = fit_teacher_models(
            X_train,
            feature_cols=teacher_feature_cols,
            cat_features=teacher_cat_features,
            target_df=y_train,
        )
        block_pred = predict_teacher_signals_for_valid(block_models, X_pred)
        oof_rows.append(block_pred)

    if not oof_rows:
        raise RuntimeError("Teacher OOF signals gagal dibuat. Cek ukuran split / warmup.")

    oof_df = pd.concat(oof_rows, axis=0, ignore_index=True)
    oof_df = oof_df.drop_duplicates("match_id", keep="last")
    return oof_df

teacher_signal_train_oof = build_teacher_oof_signals(
    train_fold_teacher_df=train_fold_teacher_df,
    train_fold_targets=y_train_fold,
    n_temporal_blocks=3,
)
print("[INFO] teacher_signal_train_oof shape:", teacher_signal_train_oof.shape)
display(teacher_signal_train_oof.head())


## 12. Student Distill Models

Di tahap ini student tetap hanya memakai fitur legal, tetapi targetnya adalah **teacher signals** yang sudah dibuat secara temporal dan fair.  
Kita sengaja memakai model-model kecil terpisah supaya pipeline tetap mudah diaudit dan tidak berubah jadi custom training loop yang terlalu rumit.


In [ ]:
distill_train_df = train_fold_student_df.merge(teacher_signal_train_oof, on="match_id", how="inner")
print("[INFO] distill_train_df shape:", distill_train_df.shape)

def fit_student_distill_models(distill_train_df: pd.DataFrame, feature_cols: list[str], cat_features: list[str]) -> dict:
    aligned_feature_cols, aligned_cat_features = _align_feature_columns(
        feature_df=distill_train_df,
        feature_cols=feature_cols,
        cat_features=cat_features,
        context="fit_student_distill_models",
    )
    X = distill_train_df[aligned_feature_cols].copy()

    soft_reg_params = goal_reg_params_map["rmse"].copy()
    soft_reg_params["iterations"] = 1200
    soft_reg_params["verbose"] = 200

    models = {
        "feature_cols": aligned_feature_cols,
        "cat_features": aligned_cat_features,
        "goal_a": fit_regressor(X, distill_train_df["teacher_goal_a_hat"], aligned_cat_features, soft_reg_params),
        "goal_b": fit_regressor(X, distill_train_df["teacher_goal_b_hat"], aligned_cat_features, soft_reg_params),
        "total": fit_regressor(X, distill_train_df["teacher_total_hat"], aligned_cat_features, soft_reg_params),
        "gd": fit_regressor(X, distill_train_df["teacher_gd_hat"], aligned_cat_features, soft_reg_params),
        "p_a_win": fit_regressor(X, distill_train_df["teacher_p_a_win"], aligned_cat_features, soft_reg_params),
        "p_draw": fit_regressor(X, distill_train_df["teacher_p_draw"], aligned_cat_features, soft_reg_params),
        "p_b_win": fit_regressor(X, distill_train_df["teacher_p_b_win"], aligned_cat_features, soft_reg_params),
        "p_a_blowout_5": fit_regressor(X, distill_train_df["teacher_p_a_blowout_5plus"], aligned_cat_features, soft_reg_params),
        "p_b_blowout_5": fit_regressor(X, distill_train_df["teacher_p_b_blowout_5plus"], aligned_cat_features, soft_reg_params),
        "p_a_blowout_7": fit_regressor(X, distill_train_df["teacher_p_a_blowout_7plus"], aligned_cat_features, soft_reg_params),
        "p_b_blowout_7": fit_regressor(X, distill_train_df["teacher_p_b_blowout_7plus"], aligned_cat_features, soft_reg_params),
        "p_high_total": fit_regressor(X, distill_train_df["teacher_p_high_total"], aligned_cat_features, soft_reg_params),
    }
    return models

student_distill_models = fit_student_distill_models(
    distill_train_df=distill_train_df,
    feature_cols=student_feature_cols,
    cat_features=student_cat_features,
)

## 13. Student Distill Validation

Sekarang kita lihat apakah student yang meniru teacher signals bisa memberi raw output yang lebih berguna daripada student anchor hard-label-only.


In [ ]:
def predict_student_distill_outputs(feature_df: pd.DataFrame, models: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_feature_cols, _ = _align_feature_columns(
        feature_df=feature_df,
        feature_cols=models["feature_cols"],
        cat_features=None,
        context="predict_student_distill_outputs",
    )
    X = feature_df[model_feature_cols].copy()

    raw_df = pd.DataFrame({
        "match_id": feature_df["match_id"].values,
        "tournament": feature_df["tournament"].values,
        "actual_team_a_goals": feature_df["team_a_goals"].values if "team_a_goals" in feature_df.columns else np.nan,
        "actual_team_b_goals": feature_df["team_b_goals"].values if "team_b_goals" in feature_df.columns else np.nan,
        "pred_goal_a_cont": models["goal_a"].predict(X),
        "pred_goal_b_cont": models["goal_b"].predict(X),
        "pred_total_cont": models["total"].predict(X),
        "pred_gd_cont": models["gd"].predict(X),
    })

    p_a_win = np.clip(models["p_a_win"].predict(X), 1e-6, 1.0)
    p_draw = np.clip(models["p_draw"].predict(X), 1e-6, 1.0)
    p_b_win = np.clip(models["p_b_win"].predict(X), 1e-6, 1.0)
    row_sum = p_a_win + p_draw + p_b_win

    raw_df["pred_outcome_proba_0"] = p_a_win / row_sum
    raw_df["pred_outcome_proba_1"] = p_draw / row_sum
    raw_df["pred_outcome_proba_2"] = p_b_win / row_sum

    tail_prob_df = pd.DataFrame({
        "match_id": feature_df["match_id"].values,
        "prob_a_blowout_5plus": np.clip(models["p_a_blowout_5"].predict(X), 1e-6, 1.0),
        "prob_b_blowout_5plus": np.clip(models["p_b_blowout_5"].predict(X), 1e-6, 1.0),
        "prob_a_blowout_7plus": np.clip(models["p_a_blowout_7"].predict(X), 1e-6, 1.0),
        "prob_b_blowout_7plus": np.clip(models["p_b_blowout_7"].predict(X), 1e-6, 1.0),
        "prob_high_total": np.clip(models["p_high_total"].predict(X), 1e-6, 1.0),
    })
    return raw_df, tail_prob_df

student_distill_raw_valid, student_distill_tail_valid = predict_student_distill_outputs(
    valid_fold_student_df,
    student_distill_models,
)

decoder_grid_student_distill, best_decoder_student_distill, pred_student_distill = tune_decoder_from_raw_outputs(
    student_distill_raw_valid,
    anchor_decoder_grid,
    scoreline_prior_lookup,
)

valid_pred_match_student_distill = student_distill_raw_valid.merge(pred_student_distill, on="match_id", how="left")
valid_pred_match_student_distill.to_csv(PRED_DIR / "valid_pred_match_student_distill.csv", index=False)
decoder_grid_student_distill.to_csv(SUM_DIR / "decoder_grid_student_distill.csv", index=False)

awmae_student_distill = awmae_score(
    valid_pred_match_student_distill["actual_team_a_goals"],
    valid_pred_match_student_distill["actual_team_b_goals"],
    valid_pred_match_student_distill["pred_team_a_goals"],
    valid_pred_match_student_distill["pred_team_b_goals"],
    valid_pred_match_student_distill["tournament"],
)

print(f"[RESULT] student_distill valid AW-MAE = {awmae_student_distill:.6f}")

## 14. Student Tail-Aware Validation

Ini variant utama EXP04B.  
Student tetap legal dan freeze-safe, tetapi decoder diberi sinyal **directional blowout**:

- apakah team A berpotensi menang besar,
- apakah team B berpotensi menang besar,
- apakah total gol berpotensi tinggi.

Tuning dibuat **staged** supaya runtime tetap aman.


In [ ]:
def _build_directional_candidate_grid(max_goals: int, scoreline_prior: dict, eps: float = 1e-12) -> dict:
    a_vals = np.arange(max_goals + 1, dtype=np.int16)
    b_vals = np.arange(max_goals + 1, dtype=np.int16)
    cand_a, cand_b = np.meshgrid(a_vals, b_vals, indexing="ij")
    cand_a = cand_a.reshape(-1)
    cand_b = cand_b.reshape(-1)
    cand_total = cand_a + cand_b
    cand_gd = cand_a - cand_b
    cand_outcome = np.where(cand_a > cand_b, 0, np.where(cand_a == cand_b, 1, 2)).astype(np.int8)
    prior_vals = np.array([float(scoreline_prior.get((int(a), int(b)), eps)) for a, b in zip(cand_a, cand_b)], dtype=np.float64)
    prior_vals = np.clip(prior_vals, eps, 1.0)

    return {
        "cand_a": cand_a,
        "cand_b": cand_b,
        "cand_total": cand_total,
        "cand_gd": cand_gd,
        "cand_outcome": cand_outcome,
        "prior_vals": prior_vals,
        "is_a5": (cand_gd >= 5),
        "is_b5": (cand_gd <= -5),
        "is_a7": (cand_gd >= 7),
        "is_b7": (cand_gd <= -7),
        "is_ht": (cand_total >= 6),
    }

def decode_directional_tail_batch(
    raw_df: pd.DataFrame,
    tail_prob_df: pd.DataFrame,
    decoder_params: dict,
    scoreline_prior: dict,
    eps: float = 1e-12,
) -> pd.DataFrame:
    grid = _build_directional_candidate_grid(int(decoder_params["MAX_GOALS"]), scoreline_prior, eps=eps)

    cand_a = grid["cand_a"][None, :].astype(np.float64)
    cand_b = grid["cand_b"][None, :].astype(np.float64)
    cand_total = grid["cand_total"][None, :].astype(np.float64)
    cand_gd = grid["cand_gd"][None, :].astype(np.float64)
    cand_outcome = grid["cand_outcome"]
    prior_cost = -np.log(grid["prior_vals"] + eps)[None, :]

    pred_goal_a = raw_df["pred_goal_a_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_goal_b = raw_df["pred_goal_b_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_total = raw_df["pred_total_cont"].to_numpy(dtype=np.float64)[:, None]
    pred_gd = raw_df["pred_gd_cont"].to_numpy(dtype=np.float64)[:, None]

    outcome_probs = raw_df[["pred_outcome_proba_0", "pred_outcome_proba_1", "pred_outcome_proba_2"]].to_numpy(dtype=np.float64)
    outcome_probs = np.clip(outcome_probs, eps, 1.0)
    outcome_cost = -np.log(outcome_probs[:, cand_outcome] + eps)

    base_cost = (
        float(decoder_params["w_direct"]) * (np.abs(cand_a - pred_goal_a) + np.abs(cand_b - pred_goal_b))
        + float(decoder_params["w_total"]) * np.abs(cand_total - pred_total)
        + float(decoder_params["w_gd"]) * np.abs(cand_gd - pred_gd)
        + float(decoder_params["w_outcome"]) * outcome_cost
        + float(decoder_params["w_prior"]) * prior_cost
    )

    p_a5 = np.clip(tail_prob_df["prob_a_blowout_5plus"].to_numpy(dtype=np.float64), eps, 1.0 - eps)[:, None]
    p_b5 = np.clip(tail_prob_df["prob_b_blowout_5plus"].to_numpy(dtype=np.float64), eps, 1.0 - eps)[:, None]
    p_a7 = np.clip(tail_prob_df["prob_a_blowout_7plus"].to_numpy(dtype=np.float64), eps, 1.0 - eps)[:, None]
    p_b7 = np.clip(tail_prob_df["prob_b_blowout_7plus"].to_numpy(dtype=np.float64), eps, 1.0 - eps)[:, None]
    p_ht = np.clip(tail_prob_df["prob_high_total"].to_numpy(dtype=np.float64), eps, 1.0 - eps)[:, None]

    tail_cost = np.zeros_like(base_cost)

    for key, prob, weight_name in [
        ("is_a5", p_a5, "w_a5"),
        ("is_b5", p_b5, "w_b5"),
        ("is_a7", p_a7, "w_a7"),
        ("is_b7", p_b7, "w_b7"),
        ("is_ht", p_ht, "w_ht"),
    ]:
        weight = float(decoder_params[weight_name])
        if weight <= 0:
            continue
        tail_cost += np.where(
            grid[key][None, :],
            weight * (-np.log(prob)),
            0.25 * weight * (-np.log(1.0 - prob)),
        )

    total_cost = base_cost + tail_cost
    best_idx = np.argmin(total_cost, axis=1)

    return pd.DataFrame({
        "match_id": raw_df["match_id"].values,
        "pred_team_a_goals": grid["cand_a"][best_idx].astype(int),
        "pred_team_b_goals": grid["cand_b"][best_idx].astype(int),
    })

def tune_directional_tail_decoder(
    raw_df: pd.DataFrame,
    tail_prob_df: pd.DataFrame,
    stage1_grid: dict,
    scoreline_prior: dict,
    top_n: int = 3,
) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    stage1_rows = []
    keys = list(stage1_grid.keys())
    combos = list(product(*[stage1_grid[k] for k in keys]))

    for combo in tqdm(combos, total=len(combos), desc="tune_directional_tail_stage1"):
        params = dict(zip(keys, combo))
        prior = scoreline_prior[int(params["MAX_GOALS"])]
        pred_df = decode_directional_tail_batch(raw_df, tail_prob_df, params, prior)
        score = awmae_score(
            raw_df["actual_team_a_goals"],
            raw_df["actual_team_b_goals"],
            pred_df["pred_team_a_goals"],
            pred_df["pred_team_b_goals"],
            raw_df["tournament"],
        )
        stage1_rows.append({**params, "valid_awmae": float(score), "stage": "stage1"})

    stage1_df = pd.DataFrame(stage1_rows).sort_values("valid_awmae").reset_index(drop=True)
    top_df = stage1_df.head(top_n).copy()

    refine_rows = []
    for _, base_row in top_df.iterrows():
        local_grid = {
            "MAX_GOALS": [int(base_row["MAX_GOALS"])],
            "w_direct": [float(base_row["w_direct"])],
            "w_total": [float(base_row["w_total"])],
            "w_gd": [float(base_row["w_gd"])],
            "w_outcome": [float(base_row["w_outcome"])],
            "w_prior": [float(base_row["w_prior"])],
            "w_a5": sorted(set([max(0.0, float(base_row["w_a5"]) - 0.25), float(base_row["w_a5"]), float(base_row["w_a5"]) + 0.25])),
            "w_b5": sorted(set([max(0.0, float(base_row["w_b5"]) - 0.25), float(base_row["w_b5"]), float(base_row["w_b5"]) + 0.25])),
            "w_a7": sorted(set([max(0.0, float(base_row["w_a7"]) - 0.25), float(base_row["w_a7"]), float(base_row["w_a7"]) + 0.25])),
            "w_b7": sorted(set([max(0.0, float(base_row["w_b7"]) - 0.25), float(base_row["w_b7"]), float(base_row["w_b7"]) + 0.25])),
            "w_ht": sorted(set([max(0.0, float(base_row["w_ht"]) - 0.25), float(base_row["w_ht"]), float(base_row["w_ht"]) + 0.25])),
        }
        local_keys = list(local_grid.keys())
        local_combos = list(product(*[local_grid[k] for k in local_keys]))

        for combo in local_combos:
            params = dict(zip(local_keys, combo))
            prior = scoreline_prior[int(params["MAX_GOALS"])]
            pred_df = decode_directional_tail_batch(raw_df, tail_prob_df, params, prior)
            score = awmae_score(
                raw_df["actual_team_a_goals"],
                raw_df["actual_team_b_goals"],
                pred_df["pred_team_a_goals"],
                pred_df["pred_team_b_goals"],
                raw_df["tournament"],
            )
            refine_rows.append({**params, "valid_awmae": float(score), "stage": "stage2"})

    refine_df = pd.DataFrame(refine_rows).sort_values("valid_awmae").reset_index(drop=True)
    all_df = pd.concat([stage1_df, refine_df], axis=0, ignore_index=True).drop_duplicates(
        subset=["MAX_GOALS", "w_direct", "w_total", "w_gd", "w_outcome", "w_prior", "w_a5", "w_b5", "w_a7", "w_b7", "w_ht"],
        keep="first",
    ).sort_values("valid_awmae").reset_index(drop=True)

    best = all_df.iloc[0].to_dict()
    pred_df = decode_directional_tail_batch(raw_df, tail_prob_df, best, scoreline_prior[int(best["MAX_GOALS"])])
    return all_df, best, pred_df


In [ ]:
tail_stage1_grid = {
    "MAX_GOALS": [7, 9],
    "w_direct": [0.75, 1.0],
    "w_total": [1.0],
    "w_gd": [1.0],
    "w_outcome": [1.5, 2.0],
    "w_prior": [0.2, 0.5],
    "w_a5": [0.0, 0.5],
    "w_b5": [0.0, 0.5],
    "w_a7": [0.0, 0.5],
    "w_b7": [0.0, 0.5],
    "w_ht": [0.0, 0.5],
}

decoder_grid_student_tailaware, best_decoder_student_tailaware, pred_student_tailaware = tune_directional_tail_decoder(
    raw_df=student_distill_raw_valid,
    tail_prob_df=student_distill_tail_valid,
    stage1_grid=tail_stage1_grid,
    scoreline_prior=scoreline_prior_lookup,
    top_n=3,
)

valid_pred_match_student_tailaware = student_distill_raw_valid.merge(pred_student_tailaware, on="match_id", how="left")
valid_pred_match_student_tailaware = valid_pred_match_student_tailaware.merge(student_distill_tail_valid, on="match_id", how="left")
valid_pred_match_student_tailaware.to_csv(PRED_DIR / "valid_pred_match_student_tailaware.csv", index=False)
decoder_grid_student_tailaware.to_csv(SUM_DIR / "decoder_grid_student_tailaware.csv", index=False)

awmae_student_tailaware = awmae_score(
    valid_pred_match_student_tailaware["actual_team_a_goals"],
    valid_pred_match_student_tailaware["actual_team_b_goals"],
    valid_pred_match_student_tailaware["pred_team_a_goals"],
    valid_pred_match_student_tailaware["pred_team_b_goals"],
    valid_pred_match_student_tailaware["tournament"],
)

print(f"[RESULT] student_tailaware valid AW-MAE = {awmae_student_tailaware:.6f}")
display(pd.DataFrame([best_decoder_student_tailaware]))


## 15. Optional Student Blend

Eksperimen ini sengaja tidak mengandalkan blend besar.  
Kalau diperlukan, blend kecil bisa dicoba belakangan, tetapi default notebook ini fokus pada empat variant utama agar runtime tetap aman.


## 16. Variant Comparison Table

Sekarang kita ringkas semua variant utama di notebook yang sama:

- `student_anchor`
- `teacher_upper_bound`
- `student_distill`
- `student_tailaware`


In [ ]:
variant_metrics = pd.DataFrame([
    {
        "variant_name": "student_anchor",
        "submission_safe": True,
        "valid_awmae": awmae_student_anchor,
        "notes": f"objective={SAFE_GOAL_OBJECTIVE_NAME}, hard labels only",
    },
    {
        "variant_name": "teacher_upper_bound",
        "submission_safe": False,
        "valid_awmae": awmae_teacher_upper_bound,
        "notes": "analysis only, uses teacher-only official train features",
    },
    {
        "variant_name": "student_distill",
        "submission_safe": True,
        "valid_awmae": awmae_student_distill,
        "notes": "student legal features, distilled teacher soft signals",
    },
    {
        "variant_name": "student_tailaware",
        "submission_safe": True,
        "valid_awmae": awmae_student_tailaware,
        "notes": "student distill + directional tail-aware decoder",
    },
]).sort_values("valid_awmae").reset_index(drop=True)

variant_metrics.to_csv(SUM_DIR / "variant_metrics.csv", index=False)
display(variant_metrics)

plt.figure(figsize=(8, 4))
sns.barplot(data=variant_metrics, x="variant_name", y="valid_awmae")
plt.xticks(rotation=20)
plt.title("Validation AW-MAE Comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "awmae_variant_comparison.png", dpi=150)
plt.show()


## 17. Headroom Analysis

Di sini kita lihat dua hal:

1. seberapa tinggi headroom teacher dibanding student anchor,
2. seberapa banyak gap itu berhasil dipindahkan ke student distill / tailaware.


In [ ]:
best_student_safe_awmae = variant_metrics.loc[variant_metrics["submission_safe"], "valid_awmae"].min()

headroom_table = pd.DataFrame([
    {
        "comparison": "teacher_upper_bound vs student_anchor",
        "gap": awmae_student_anchor - awmae_teacher_upper_bound,
    },
    {
        "comparison": "student_distill vs student_anchor",
        "gap": awmae_student_anchor - awmae_student_distill,
    },
    {
        "comparison": "student_tailaware vs student_anchor",
        "gap": awmae_student_anchor - awmae_student_tailaware,
    },
    {
        "comparison": "teacher_upper_bound vs best_student_safe",
        "gap": best_student_safe_awmae - awmae_teacher_upper_bound,
    },
])
display(headroom_table)

plt.figure(figsize=(8, 4))
sns.barplot(data=headroom_table, x="comparison", y="gap")
plt.xticks(rotation=20)
plt.title("Headroom Gap Analysis")
plt.tight_layout()
plt.savefig(FIG_DIR / "headroom_gap_analysis.png", dpi=150)
plt.show()


## 18. Raw Signal Analysis

Selain AW-MAE, kita lihat juga signal mentahnya supaya lebih jelas apakah improvement datang dari raw model signal atau hanya dari perubahan decoder.


In [ ]:
def summarize_raw_signal(raw_df: pd.DataFrame, name: str) -> dict:
    pred_outcome = raw_df[["pred_outcome_proba_0", "pred_outcome_proba_1", "pred_outcome_proba_2"]].values.argmax(axis=1)
    true_outcome = [
        _outcome(a, b)
        for a, b in zip(raw_df["actual_team_a_goals"], raw_df["actual_team_b_goals"])
    ]
    return {
        "variant_name": name,
        "outcome_acc": accuracy_score(true_outcome, pred_outcome),
        "mae_goal_a": mean_absolute_error(raw_df["actual_team_a_goals"], raw_df["pred_goal_a_cont"]),
        "mae_goal_b": mean_absolute_error(raw_df["actual_team_b_goals"], raw_df["pred_goal_b_cont"]),
        "mae_total": mean_absolute_error(
            raw_df["actual_team_a_goals"] + raw_df["actual_team_b_goals"],
            raw_df["pred_total_cont"],
        ),
        "mae_gd": mean_absolute_error(
            raw_df["actual_team_a_goals"] - raw_df["actual_team_b_goals"],
            raw_df["pred_gd_cont"],
        ),
    }

raw_signal_table = pd.DataFrame([
    summarize_raw_signal(student_anchor_raw_valid, "student_anchor"),
    summarize_raw_signal(teacher_raw_valid, "teacher_upper_bound"),
    summarize_raw_signal(student_distill_raw_valid, "student_distill"),
])
display(raw_signal_table)


## 19. Tail / Blowout Analysis Wajib

Ini bagian paling penting dari EXP04B.  
Kita lihat apakah branch distillation + directional tail-aware benar-benar membantu di pertandingan yang:

- margin kemenangannya besar,
- skor maksimumnya tinggi,
- atau memang termasuk blowout context.


In [ ]:
valid_eval_base = valid_fold_base[["match_id", "team_a_goals", "team_b_goals", "gender", "neutral", "tournament"]].copy()
valid_eval_base["goal_diff"] = valid_eval_base["team_a_goals"] - valid_eval_base["team_b_goals"]
valid_eval_base["total_goals"] = valid_eval_base["team_a_goals"] + valid_eval_base["team_b_goals"]
valid_eval_base["extreme_abs_gd_5"] = (valid_eval_base["goal_diff"].abs() >= 5).astype(int)
valid_eval_base["extreme_abs_gd_7"] = (valid_eval_base["goal_diff"].abs() >= 7).astype(int)
valid_eval_base["extreme_max_goal_6"] = (valid_eval_base[["team_a_goals", "team_b_goals"]].max(axis=1) >= 6).astype(int)

variant_pred_lookup = {
    "student_anchor": valid_pred_match_student_anchor[["match_id", "pred_team_a_goals", "pred_team_b_goals"]],
    "teacher_upper_bound": valid_pred_match_teacher_upper_bound[["match_id", "pred_team_a_goals", "pred_team_b_goals"]],
    "student_distill": valid_pred_match_student_distill[["match_id", "pred_team_a_goals", "pred_team_b_goals"]],
    "student_tailaware": valid_pred_match_student_tailaware[["match_id", "pred_team_a_goals", "pred_team_b_goals"]],
}

def subgroup_awmae(eval_df: pd.DataFrame, subgroup_col: str) -> pd.DataFrame:
    rows = []
    for variant_name, pred_df in variant_pred_lookup.items():
        merged = eval_df.merge(pred_df, on="match_id", how="left")
        for subgroup_value, grp in merged.groupby(subgroup_col):
            score = awmae_score(
                grp["team_a_goals"], grp["team_b_goals"],
                grp["pred_team_a_goals"], grp["pred_team_b_goals"],
                grp["tournament"],
            )
            rows.append({
                "variant_name": variant_name,
                "subgroup_col": subgroup_col,
                "subgroup_value": subgroup_value,
                "awmae": float(score),
                "n_matches": int(len(grp)),
            })
    return pd.DataFrame(rows)

blowout_tables = pd.concat([
    subgroup_awmae(valid_eval_base, "extreme_abs_gd_5"),
    subgroup_awmae(valid_eval_base, "extreme_abs_gd_7"),
    subgroup_awmae(valid_eval_base, "extreme_max_goal_6"),
], axis=0, ignore_index=True)

display(blowout_tables.head(20))

plt.figure(figsize=(10, 4))
plot_df = blowout_tables[blowout_tables["subgroup_col"] == "extreme_abs_gd_5"].copy()
plot_df["label"] = plot_df["subgroup_col"] + "=" + plot_df["subgroup_value"].astype(str)
sns.barplot(data=plot_df, x="variant_name", y="awmae", hue="label")
plt.title("Blowout Subgroup Comparison (abs(goal_diff) >= 5)")
plt.tight_layout()
plt.savefig(FIG_DIR / "blowout_subgroup_comparison.png", dpi=150)
plt.show()

loss_case_df = valid_eval_base.merge(
    valid_pred_match_student_anchor[["match_id", "pred_team_a_goals", "pred_team_b_goals"]].rename(
        columns={"pred_team_a_goals": "anchor_a", "pred_team_b_goals": "anchor_b"}
    ),
    on="match_id",
    how="left",
).merge(
    valid_pred_match_student_tailaware[["match_id", "pred_team_a_goals", "pred_team_b_goals"]].rename(
        columns={"pred_team_a_goals": "tail_a", "pred_team_b_goals": "tail_b"}
    ),
    on="match_id",
    how="left",
)

loss_case_df["anchor_loss"] = [
    official_match_loss(a, b, pa, pb)
    for a, b, pa, pb in zip(
        loss_case_df["team_a_goals"], loss_case_df["team_b_goals"], loss_case_df["anchor_a"], loss_case_df["anchor_b"]
    )
]
loss_case_df["tail_loss"] = [
    official_match_loss(a, b, pa, pb)
    for a, b, pa, pb in zip(
        loss_case_df["team_a_goals"], loss_case_df["team_b_goals"], loss_case_df["tail_a"], loss_case_df["tail_b"]
    )
]
loss_case_df["tail_improvement"] = loss_case_df["anchor_loss"] - loss_case_df["tail_loss"]

top_loss_examples = loss_case_df.sort_values("anchor_loss", ascending=False).head(15)
display(top_loss_examples[[
    "match_id", "team_a_goals", "team_b_goals", "goal_diff", "total_goals",
    "anchor_a", "anchor_b", "tail_a", "tail_b",
    "anchor_loss", "tail_loss", "tail_improvement"
]])

plt.figure(figsize=(10, 5))
tmp = top_loss_examples.copy()
tmp["label"] = tmp["match_id"].astype(str)
plt.plot(tmp["label"], tmp["anchor_loss"], marker="o", label="anchor_loss")
plt.plot(tmp["label"], tmp["tail_loss"], marker="o", label="tailaware_loss")
plt.xticks(rotation=60)
plt.legend()
plt.title("Top Loss Case Examples")
plt.tight_layout()
plt.savefig(FIG_DIR / "top_loss_case_examples.png", dpi=150)
plt.show()


## 20. Additional Subgroup Analysis

Setelah fokus ke blowout subgroup, kita cek juga apakah improvement ini stabil pada subgroup umum seperti:

- `gender`
- `neutral`
- tournament besar


In [ ]:
gender_awmae = subgroup_awmae(valid_eval_base, "gender")
neutral_awmae = subgroup_awmae(valid_eval_base, "neutral")

top_tournaments = valid_eval_base["tournament"].value_counts().head(8).index.tolist()
tournament_eval_df = valid_eval_base[valid_eval_base["tournament"].isin(top_tournaments)].copy()
tournament_awmae = subgroup_awmae(tournament_eval_df, "tournament")

for df_plot, fig_name, title in [
    (gender_awmae, "subgroup_awmae_gender.png", "AW-MAE by Gender"),
    (neutral_awmae, "subgroup_awmae_neutral.png", "AW-MAE by Neutral"),
]:
    plt.figure(figsize=(9, 4))
    sns.barplot(data=df_plot, x="variant_name", y="awmae", hue="subgroup_value")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fig_name, dpi=150)
    plt.show()

display(gender_awmae)
display(neutral_awmae)
display(tournament_awmae.head(20))


## 21. Feature Importance Analysis

Di tahap ini kita bandingkan:

- teacher outcome model,
- student anchor outcome model,
- dan salah satu student distill tail model.

Tujuannya supaya kelihatan apakah teacher-only official features memang menyimpan sinyal yang tajam, dan apakah student mulai menangkap pola itu lewat distillation.


In [ ]:
teacher_outcome_imp = pd.DataFrame({
    "feature": teacher_feature_cols,
    "importance": teacher_models["outcome"].get_feature_importance(),
    "model": "teacher_outcome",
})
student_anchor_outcome_imp = pd.DataFrame({
    "feature": student_feature_cols,
    "importance": student_anchor_models["outcome"].get_feature_importance(),
    "model": "student_anchor_outcome",
})
student_distill_tail_imp = pd.DataFrame({
    "feature": student_feature_cols,
    "importance": student_distill_models["p_a_blowout_5"].get_feature_importance(),
    "model": "student_distill_tail_a5",
})

importance_compare = pd.concat(
    [
        teacher_outcome_imp.sort_values("importance", ascending=False).head(20),
        student_anchor_outcome_imp.sort_values("importance", ascending=False).head(20),
        student_distill_tail_imp.sort_values("importance", ascending=False).head(20),
    ],
    axis=0,
    ignore_index=True,
)
display(importance_compare.head(30))

plt.figure(figsize=(10, 6))
plot_imp = importance_compare.groupby(["feature", "model"], as_index=False)["importance"].sum()
top_features = plot_imp.groupby("feature")["importance"].sum().sort_values(ascending=False).head(15).index.tolist()
plot_imp = plot_imp[plot_imp["feature"].isin(top_features)].copy()
sns.barplot(data=plot_imp, x="importance", y="feature", hue="model")
plt.title("Feature Importance: Teacher vs Student")
plt.tight_layout()
plt.savefig(FIG_DIR / "feature_importance_teacher_vs_student.png", dpi=150)
plt.show()


## 22. Keputusan Eksperimen

Di bagian ini notebook harus memberi keputusan yang tegas, bukan abu-abu.

Pilihan keputusan final:

- **A** — distillation + directional tail awareness membantu nyata dan layak jadi mainline,
- **B** — distillation membantu tipis, tetapi belum cukup besar,
- **C** — distillation tidak membantu dan anchor tetap terbaik.


In [ ]:
best_safe_variant_name = variant_metrics[variant_metrics["submission_safe"]].iloc[0]["variant_name"]
best_safe_awmae = float(variant_metrics[variant_metrics["submission_safe"]].iloc[0]["valid_awmae"])

if best_safe_variant_name == "student_tailaware" and (awmae_student_anchor - best_safe_awmae) >= 0.05:
    experiment_decision = "A — Distillation + directional tail awareness membantu nyata dan layak jadi mainline."
elif best_safe_variant_name in ["student_distill", "student_tailaware"] and (awmae_student_anchor - best_safe_awmae) > 0:
    experiment_decision = "B — Distillation membantu tipis, tetapi belum cukup besar."
else:
    experiment_decision = "C — Distillation tidak membantu dan anchor tetap terbaik."

print("[FINAL DECISION]")
print(experiment_decision)


## 23. Retrain Best Safe Variant pada Full Train

Setelah variant terbaik dipilih di validation, kita retrain **hanya student variant terbaik** pada full train.  
Teacher tetap hanya alat bantu training / analysis, bukan sumber submission final.


In [ ]:
full_train_static = train_static_features.copy()
full_train_history, cutoff_team_states_test, cutoff_h2h_states_test = build_train_history_features_full(full_train_static)
full_train_student_df = full_train_static.merge(full_train_history, on="match_id", how="left")

full_test_history_freeze = build_future_history_features_freeze(
    test_static_features,
    cutoff_team_states_test,
    cutoff_h2h_states_test,
)
test_student_features = test_static_features.merge(full_test_history_freeze, on="match_id", how="left")

y_full_train = pd.DataFrame({
    "match_id": full_train_student_df["match_id"],
    "y_goal_a": full_train_student_df["team_a_goals"],
    "y_goal_b": full_train_student_df["team_b_goals"],
    "y_total": full_train_student_df["team_a_goals"] + full_train_student_df["team_b_goals"],
    "y_gd": full_train_student_df["team_a_goals"] - full_train_student_df["team_b_goals"],
    "y_outcome": [_outcome(a, b) for a, b in zip(full_train_student_df["team_a_goals"], full_train_student_df["team_b_goals"])],
    "is_team_a_blowout_5plus": ((full_train_student_df["team_a_goals"] - full_train_student_df["team_b_goals"]) >= 5).astype(int),
    "is_team_b_blowout_5plus": ((full_train_student_df["team_a_goals"] - full_train_student_df["team_b_goals"]) <= -5).astype(int),
    "is_team_a_blowout_7plus": ((full_train_student_df["team_a_goals"] - full_train_student_df["team_b_goals"]) >= 7).astype(int),
    "is_team_b_blowout_7plus": ((full_train_student_df["team_a_goals"] - full_train_student_df["team_b_goals"]) <= -7).astype(int),
    "is_high_total": ((full_train_student_df["team_a_goals"] + full_train_student_df["team_b_goals"]) >= 6).astype(int),
})

if best_safe_variant_name == "student_anchor":
    best_safe_models = fit_hard_model_block(
        full_train_student_df,
        feature_cols=student_feature_cols,
        cat_features=student_cat_features,
        target_df=y_full_train,
    )
elif best_safe_variant_name in ["student_distill", "student_tailaware"]:
    # Build full teacher feature frame = legal student features + teacher-only block.
    full_train_teacher_only = build_teacher_feature_set(train_match_base)
    full_train_teacher_df = full_train_student_df.merge(full_train_teacher_only, on="match_id", how="left")

    # Extra guard to fail early with a clearer message if any teacher feature is still missing.
    missing_teacher_cols = [c for c in teacher_feature_cols if c not in full_train_teacher_df.columns]
    if missing_teacher_cols:
        raise KeyError(f"Missing teacher columns on full retrain frame: {missing_teacher_cols[:20]}")

    teacher_models_full = fit_teacher_models(
        full_train_teacher_df,
        feature_cols=teacher_feature_cols,
        cat_features=teacher_cat_features,
        target_df=y_full_train,
    )
    teacher_signal_full_train = predict_teacher_signals_for_valid(teacher_models_full, full_train_teacher_df)
    distill_full_train_df = full_train_student_df.merge(teacher_signal_full_train, on="match_id", how="inner")
    best_safe_models = fit_student_distill_models(
        distill_train_df=distill_full_train_df,
        feature_cols=student_feature_cols,
        cat_features=student_cat_features,
    )
    best_teacher_models_full = teacher_models_full
else:
    raise ValueError(f"Variant aman terbaik tidak dikenali: {best_safe_variant_name}")

print(f"[OK] retrain selesai untuk best safe variant: {best_safe_variant_name}")

## 24. Test Inference

Inference test tetap **freeze-safe only**.  
Teacher tidak boleh dipakai langsung untuk submission final. Kalau variant terbaik butuh distill/tail signals, student harus memproduksi semuanya sendiri dari fitur legal.


In [ ]:
if best_safe_variant_name == "student_anchor":
    test_raw_outputs = predict_raw_outputs_for_df(test_student_features, best_safe_models)
    test_pred_match_best = decode_anchor_batch_from_raw(
        test_raw_outputs,
        best_decoder_anchor,
        scoreline_prior_lookup[int(best_decoder_anchor["MAX_GOALS"])],
    )
elif best_safe_variant_name == "student_distill":
    test_raw_outputs, test_tail_probs = predict_student_distill_outputs(test_student_features, best_safe_models)
    test_pred_match_best = decode_anchor_batch_from_raw(
        test_raw_outputs,
        best_decoder_student_distill,
        scoreline_prior_lookup[int(best_decoder_student_distill["MAX_GOALS"])],
    )
elif best_safe_variant_name == "student_tailaware":
    test_raw_outputs, test_tail_probs = predict_student_distill_outputs(test_student_features, best_safe_models)
    test_pred_match_best = decode_directional_tail_batch(
        raw_df=test_raw_outputs,
        tail_prob_df=test_tail_probs,
        decoder_params=best_decoder_student_tailaware,
        scoreline_prior=scoreline_prior_lookup[int(best_decoder_student_tailaware["MAX_GOALS"])],
    )
else:
    raise ValueError(f"Variant aman terbaik tidak dikenali: {best_safe_variant_name}")

test_pred_match_best = test_pred_match_best.merge(
    test_student_features[["match_id", "team_a", "team_b"]],
    on="match_id",
    how="left",
)

test_pred_match_best = test_pred_match_best[
    ["match_id", "team_a", "team_b", "pred_team_a_goals", "pred_team_b_goals"]
].copy()

test_pred_match_best.to_csv(PRED_DIR / "test_pred_match_best_safe.csv", index=False)
display(test_pred_match_best.head())


## 25. Reverse Mapping ke Submission

Prediksi match-level terbaik sekarang diubah kembali ke row-level submission format yang sesuai file kompetisi.


In [ ]:
if "test_pred_match_best" not in globals():
    print("[WARN] test_pred_match_best belum ada. Mencoba build otomatis dari model best-safe.")

    required_vars = [
        "best_safe_variant_name",
        "best_safe_models",
        "test_student_features",
        "scoreline_prior_lookup",
    ]
    missing_required = [v for v in required_vars if v not in globals()]
    if missing_required:
        raise RuntimeError(
            "Variabel prasyarat inference test belum lengkap: "
            f"{missing_required}. Jalankan ulang Cell 41 (Section 24) terlebih dahulu."
        )

    if best_safe_variant_name == "student_anchor":
        test_raw_outputs = predict_raw_outputs_for_df(test_student_features, best_safe_models)
        test_pred_match_best = decode_anchor_batch_from_raw(
            test_raw_outputs,
            best_decoder_anchor,
            scoreline_prior_lookup[int(best_decoder_anchor["MAX_GOALS"])],
        )
    elif best_safe_variant_name == "student_distill":
        test_raw_outputs, test_tail_probs = predict_student_distill_outputs(test_student_features, best_safe_models)
        test_pred_match_best = decode_anchor_batch_from_raw(
            test_raw_outputs,
            best_decoder_student_distill,
            scoreline_prior_lookup[int(best_decoder_student_distill["MAX_GOALS"])],
        )
    elif best_safe_variant_name == "student_tailaware":
        test_raw_outputs, test_tail_probs = predict_student_distill_outputs(test_student_features, best_safe_models)
        test_pred_match_best = decode_directional_tail_batch(
            raw_df=test_raw_outputs,
            tail_prob_df=test_tail_probs,
            decoder_params=best_decoder_student_tailaware,
            scoreline_prior=scoreline_prior_lookup[int(best_decoder_student_tailaware["MAX_GOALS"])],
        )
    else:
        raise ValueError(f"Variant aman terbaik tidak dikenali: {best_safe_variant_name}")

    test_pred_match_best = test_pred_match_best.merge(
        test_student_features[["match_id", "team_a", "team_b"]],
        on="match_id",
        how="left",
    )
    test_pred_match_best = test_pred_match_best[
        ["match_id", "team_a", "team_b", "pred_team_a_goals", "pred_team_b_goals"]
    ].copy()

submission_exp04b = match_predictions_to_submission(
    test_row_df=test,
    pred_match_df=test_pred_match_best,
    canonical_team_a_col="team_a",
    canonical_team_b_col="team_b",
    pred_a_col="pred_team_a_goals",
    pred_b_col="pred_team_b_goals",
)

assert list(submission_exp04b.columns) == ["Id", "team_goals", "opp_goals"]
assert len(submission_exp04b) == len(sample_submission)
assert submission_exp04b["Id"].tolist() == sample_submission["Id"].tolist()
assert submission_exp04b.isna().sum().sum() == 0

submission_exp04b.to_csv(SUB_DIR / "submission_exp04b_best_safe.csv", index=False)

print("[OK] submission_exp04b_best_safe.csv berhasil dibuat.")
display(submission_exp04b.head())

## 26. Ringkasan Hasil Eksperimen

Checklist ringkasan yang harus dijawab notebook ini:

- seberapa tinggi **teacher upper bound**?
- apakah **student distill** membantu dibanding student anchor?
- apakah **directional tail-aware decoder** membantu di blowout subgroup?
- apakah improvement itu cukup untuk mendorong branch ini jadi mainline baru?
- apakah best safe pipeline berhasil menyalip backbone lama atau belum?


In [ ]:
summary_lines = [
    f"- Best safe variant             : {best_safe_variant_name}",
    f"- Student anchor valid AW-MAE   : {awmae_student_anchor:.6f}",
    f"- Teacher upper bound AW-MAE    : {awmae_teacher_upper_bound:.6f}",
    f"- Student distill AW-MAE        : {awmae_student_distill:.6f}",
    f"- Student tailaware AW-MAE      : {awmae_student_tailaware:.6f}",
    f"- Final experiment decision     : {experiment_decision}",
]

experiment_notes = "\n".join(summary_lines)
print(experiment_notes)

with open(SUM_DIR / "experiment_notes.txt", "w", encoding="utf-8") as f:
    f.write(experiment_notes)
